<a href="https://colab.research.google.com/github/anua78484-bit/Spotify/blob/main/spotify_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Install libraries needed for interactive visualizations
!pip install -q plotly openpyxl

# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display

print("✅ Libraries loaded successfully!")


✅ Libraries loaded successfully!


In [23]:
from google.colab import files

uploaded = files.upload()


file_path = list(uploaded.keys())[0]
df = pd.read_excel("spotify dataset final.xlsm")

print("✅ Dataset loaded successfully!")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Saving spotify dataset final.xlsm to spotify dataset final (1).xlsm
✅ Dataset loaded successfully!
Rows: 953
Columns: 25


In [6]:
# Rename columns for easier analysis
df.columns = [
    "track_name",
    "artist_name",
    "artist_count",
    "released_year",
    "released_month",
    "released_day",
    "in_spotify_playlists",
    "in_spotify_charts",
    "streams",
    "in_apple_playlists",
    "in_apple_charts",
    "in_deezer_playlists",
    "in_deezer_charts",
    "in_shazam_charts",
    "bpm",
    "key",
    "mode",
    "danceability",
    "valence",
    "energy",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "cover_url"
]

print("✅ Column names updated!")

display(pd.DataFrame({
    "Column Number": range(len(df.columns)),
    "Column Name": df.columns
}))

✅ Column names updated!


,Column Number,Column Name
0,0,track_name
1,1,artist_name
2,2,artist_count
3,3,released_year
4,4,released_month
5,5,released_day
6,6,in_spotify_playlists
7,7,in_spotify_charts
8,8,streams
9,9,in_apple_playlists


In [7]:
print("========== DATASET CHECK ==========")

print(f"Number of tracks: {len(df):,}")
print(f"Number of columns: {len(df.columns)}")

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
display(df.dtypes)

========== DATASET CHECK ==========
Number of tracks: 953
Number of columns: 25

Missing values:
track_name                0
artist_name               0
artist_count              0
released_year             0
released_month            0
released_day              0
in_spotify_playlists      0
in_spotify_charts         0
streams                   0
in_apple_playlists        0
in_apple_charts           0
in_deezer_playlists       0
in_deezer_charts          0
in_shazam_charts         50
bpm                       0
key                      95
mode                      0
danceability              0
valence                   0
energy                    0
acousticness              0
instrumentalness          0
liveness                  0
speechiness               0
cover_url               953
dtype: int64

Duplicate rows:
0

Data types:


,0
track_name,object
artist_name,object
artist_count,int64
released_year,int64
released_month,int64
released_day,int64
in_spotify_playlists,int64
in_spotify_charts,int64
streams,object
in_apple_playlists,int64


In [8]:
# Audio features needed for Question 1
audio_features = [
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "bpm"
]

# Convert only the variables needed for Question 1
df["streams"] = pd.to_numeric(df["streams"], errors="coerce")

for feature in audio_features:
    df[feature] = pd.to_numeric(df[feature], errors="coerce")

# Remove rows ONLY if streams or Q1 audio features are missing
df_q1 = df.dropna(subset=["streams"] + audio_features).copy()

print("✅ Q1 analysis dataset prepared!")
print(f"Original tracks: {len(df):,}")
print(f"Tracks used for Q1: {len(df_q1):,}")
print(f"Tracks excluded: {len(df) - len(df_q1):,}")

✅ Q1 analysis dataset prepared!
Original tracks: 953
Tracks used for Q1: 952
Tracks excluded: 1


In [9]:
# Create logarithmic stream count
df_q1["log_streams"] = np.log10(df["streams"])

print("✅ Logarithmic stream variable created.")

display(
    df_q1[["streams", "log_streams"]].describe()
)

✅ Logarithmic stream variable created.


,streams,log_streams
count,9.520000e+02,952.000000
mean,5.141374e+08,8.471630
std,5.668569e+08,0.497829
min,2.762000e+03,3.441224
25%,1.416362e+08,8.151174
50%,2.905309e+08,8.463192
75%,6.738690e+08,8.828575
max,3.703895e+09,9.568659


In [10]:
# Calculate Pearson correlation between each audio feature and streams

correlations = (
    df_q1[audio_features + ["streams"]]
    .corr()["streams"]
    .drop("streams")
    .sort_values(key=abs, ascending=False)
)

# Create a clean results table
correlation_table = pd.DataFrame({
    "Audio Feature": correlations.index,
    "Correlation": correlations.values
})

correlation_table["Absolute Correlation"] = (
    correlation_table["Correlation"].abs()
)

correlation_table = correlation_table.sort_values(
    "Absolute Correlation",
    ascending=False
)

print("🎵 AUDIO FEATURE CORRELATION WITH STREAMS")
display(correlation_table.round(3))

🎵 AUDIO FEATURE CORRELATION WITH STREAMS


,Audio Feature,Correlation,Absolute Correlation
0,speechiness,-0.112,0.112
1,danceability,-0.105,0.105
2,liveness,-0.048,0.048
3,instrumentalness,-0.045,0.045
4,valence,-0.041,0.041
5,energy,-0.026,0.026
6,acousticness,-0.004,0.004
7,bpm,-0.002,0.002


In [11]:
# Sort for horizontal visualization
plot_data = correlation_table.sort_values(
    "Absolute Correlation"
)

fig = px.bar(
    plot_data,
    x="Absolute Correlation",
    y="Audio Feature",
    orientation="h",
    text="Correlation",
    title="Which Audio Features Are Most Associated With Spotify Streams?",
    labels={
        "Absolute Correlation": "Strength of Correlation",
        "Audio Feature": ""
    },
    template="plotly_white"
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside",
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Correlation: %{text:.3f}<br>"
        "Absolute strength: %{x:.3f}"
        "<extra></extra>"
    )
)

fig.update_layout(
    height=600,
    title_font_size=24,
    font=dict(size=14),
    margin=dict(l=100, r=80, t=100, b=60)
)

fig.show()

In [13]:
# Define high-stream songs as the top 25%
high_stream_threshold = df["streams"].quantile(0.75)

df["stream_group"] = np.where(
    df["streams"] >= high_stream_threshold,
    "High Streams",
    "Lower Streams"
)

print(
    f"High-stream threshold: "
    f"{high_stream_threshold:,.0f} streams"
)

print("\nNumber of tracks in each group:")
print(df["stream_group"].value_counts())

High-stream threshold: 673,869,022 streams

Number of tracks in each group:
stream_group
Lower Streams    715
High Streams     238
Name: count, dtype: int64


In [14]:
# Calculate average audio features for each stream group

group_means = (
    df.groupby("stream_group")[audio_features]
    .mean()
    .T
)

group_means = group_means[
    ["High Streams", "Lower Streams"]
]

print("🎵 AVERAGE AUDIO FEATURES")
display(group_means.round(2))


🎵 AVERAGE AUDIO FEATURES


stream_group,High Streams,Lower Streams
danceability,64.58,67.76
energy,63.82,64.43
valence,48.80,52.31
acousticness,26.88,27.12
instrumentalness,1.23,1.70
liveness,17.55,18.43
speechiness,8.08,10.82
bpm,121.87,122.76


In [15]:
# Convert table into a format Plotly can use
comparison_data = group_means.reset_index()

comparison_data = comparison_data.rename(
    columns={"index": "Audio Feature"}
)

comparison_long = comparison_data.melt(
    id_vars="Audio Feature",
    var_name="Stream Group",
    value_name="Average Value"
)

fig = px.bar(
    comparison_long,
    x="Audio Feature",
    y="Average Value",
    color="Stream Group",
    barmode="group",
    title="Audio Profile: High-Stream vs Lower-Stream Tracks",
    labels={
        "Average Value": "Average Audio Feature Value",
        "Audio Feature": ""
    },
    template="plotly_white"
)

fig.update_layout(
    height=600,
    title_font_size=24,
    font=dict(size=14),
    xaxis_tickangle=-30
)

fig.show()


In [16]:
fig = px.scatter(
    df,
    x="danceability",
    y="streams",
    color="stream_group",
    hover_name="track_name",
    hover_data={
        "artist_name": True,
        "danceability": ":.1f",
        "energy": ":.1f",
        "streams": ":,.0f",
        "stream_group": True
    },
    log_y=True,
    title="Does Danceability Relate to Spotify Streams?",
    labels={
        "danceability": "Danceability (%)",
        "streams": "Spotify Streams",
        "stream_group": "Stream Category"
    },
    template="plotly_white"
)

fig.update_layout(
    height=650,
    title_font_size=25,
    font=dict(size=14)
)

fig.show()

In [17]:
fig = px.scatter(
    df,
    x="speechiness",
    y="streams",
    color="stream_group",
    hover_name="track_name",
    hover_data={
        "artist_name": True,
        "speechiness": ":.1f",
        "danceability": ":.1f",
        "streams": ":,.0f",
        "stream_group": True
    },
    log_y=True,
    title="Does Speechiness Relate to Spotify Streams?",
    labels={
        "speechiness": "Speechiness (%)",
        "streams": "Spotify Streams",
        "stream_group": "Stream Category"
    },
    template="plotly_white"
)

fig.update_layout(
    height=650,
    title_font_size=25,
    font=dict(size=14)
)

fig.show()

In [18]:
# Features for the interactive explorer
features = [
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness"
]

fig = go.Figure()

for feature in features:

    fig.add_trace(
        go.Scatter(
            x=df[feature],
            y=df["streams"],
            mode="markers",
            visible=(feature == "danceability"),

            text=df["track_name"],

            customdata=np.column_stack([
                df["artist_name"],
                df["streams"]
            ]),

            hovertemplate=(
                "<b>%{text}</b><br>"
                "Artist: %{customdata[0]}<br>"
                "Streams: %{customdata[1]:,.0f}<br>"
                + feature.title() +
                ": %{x:.1f}<br>"
                "<extra></extra>"
            ),

            marker=dict(
                size=8,
                opacity=0.65
            ),

            name=feature.title()
        )
    )

# Create dropdown buttons
buttons = []

for i, feature in enumerate(features):

    visibility = [False] * len(features)
    visibility[i] = True

    buttons.append(
        dict(
            label=feature.title(),
            method="update",
            args=[
                {
                    "visible": visibility
                },
                {
                    "title":
                    f"How Does {feature.title()} Relate to Spotify Streams?",

                    "xaxis": {
                        "title":
                        f"{feature.title()} (%)"
                    }
                }
            ]
        )
    )

fig.update_layout(

    title="Spotify Audio Feature Explorer",

    xaxis=dict(
        title="Danceability (%)"
    ),

    yaxis=dict(
        title="Spotify Streams",
        type="log"
    ),

    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1.0,
            xanchor="right",
            y=1.15,
            yanchor="top"
        )
    ],

    height=700,

    template="plotly_white",

    font=dict(size=14)
)

fig.show()


In [19]:
strongest_feature = correlation_table.iloc[0]["Audio Feature"]
strongest_correlation = correlation_table.iloc[0]["Correlation"]

print("🏆 STRONGEST AUDIO FEATURE")
print("--------------------------------")
print(f"Feature: {strongest_feature}")
print(f"Correlation: {strongest_correlation:.3f}")

if strongest_correlation < 0:
    direction = "negative"
else:
    direction = "positive"

print(f"Direction: {direction}")

🏆 STRONGEST AUDIO FEATURE
--------------------------------
Feature: speechiness
Correlation: -0.112
Direction: negative


In [20]:
second_feature = correlation_table.iloc[1]["Audio Feature"]
second_corr = correlation_table.iloc[1]["Correlation"]

print("🎵 QUESTION 1 — KEY FINDINGS")
print("=" * 50)

print(
    f"1. {strongest_feature.title()} has the strongest "
    f"linear relationship with streams "
    f"(r = {strongest_correlation:.3f})."
)

print(
    f"2. {second_feature.title()} has the second-strongest "
    f"relationship with streams "
    f"(r = {second_corr:.3f})."
)

print(
    f"3. The correlations are weak, meaning audio features "
    f"alone do not strongly explain streaming success."
)

print(
    "4. Other factors such as artist popularity, playlist "
    "exposure, promotion, release timing, and platform "
    "visibility may also be important."
)

🎵 QUESTION 1 — KEY FINDINGS
1. Speechiness has the strongest linear relationship with streams (r = -0.112).
2. Danceability has the second-strongest relationship with streams (r = -0.105).
3. The correlations are weak, meaning audio features alone do not strongly explain streaming success.
4. Other factors such as artist popularity, playlist exposure, promotion, release timing, and platform visibility may also be important.


In [21]:
print("""
╔══════════════════════════════════════════════════════════╗
║                 QUESTION 1 — CONCLUSION                  ║
╚══════════════════════════════════════════════════════════╝

Among the audio features analyzed, speechiness and
danceability show the strongest relationships with Spotify
stream counts.

However, both relationships are weak. This means that
audio characteristics alone do not strongly explain why
some songs receive more streams than others.

Therefore, streaming success is likely influenced by
additional factors beyond the audio profile, which can be
investigated in the next questions.
""")


╔══════════════════════════════════════════════════════════╗
║                 QUESTION 1 — CONCLUSION                  ║
╚══════════════════════════════════════════════════════════╝

Among the audio features analyzed, speechiness and
danceability show the strongest relationships with Spotify
stream counts.

However, both relationships are weak. This means that
audio characteristics alone do not strongly explain why
some songs receive more streams than others.

Therefore, streaming success is likely influenced by
additional factors beyond the audio profile, which can be
investigated in the next questions.



In [22]:
!pip install -q streamlit plotly openpyxl pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 91.8 MB/s eta 0:00:00


In [24]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Spotify Hit DNA",
    page_icon="🎵",
    layout="wide"
)

# ============================================================
# CUSTOM STYLE
# ============================================================

st.markdown("""
<style>

.main {
    background-color: #0b0f19;
}

h1 {
    font-size: 42px !important;
    font-weight: 800 !important;
}

h2 {
    font-size: 26px !important;
}

[data-testid="stMetricValue"] {
    font-size: 28px;
}

</style>
""", unsafe_allow_html=True)


# ============================================================
# TITLE
# ============================================================

st.title("🎵 Spotify Hit DNA")

st.markdown(
    """
    ### What audio features most strongly correlate with high stream counts?

    This interactive 3D visualization identifies the **two strongest audio
    features associated with Spotify streams** and maps them against the
    popularity of each track.
    """
)


# ============================================================
# LOAD DATA
# ============================================================

file_path = "spotify dataset final.xlsm"

df = pd.read_excel("spotify dataset final.xlsm")


# ============================================================
# RENAME COLUMNS
# ============================================================

df.columns = [
    "track_name",
    "artist_name",
    "artist_count",
    "released_year",
    "released_month",
    "released_day",
    "in_spotify_playlists",
    "in_spotify_charts",
    "streams",
    "in_apple_playlists",
    "in_apple_charts",
    "in_deezer_playlists",
    "in_deezer_charts",
    "in_shazam_charts",
    "bpm",
    "key",
    "mode",
    "danceability",
    "valence",
    "energy",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "cover_url"
]


# ============================================================
# AUDIO FEATURES
# ============================================================

audio_features = [
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "bpm"
]


# ============================================================
# CONVERT NUMERIC DATA
# ============================================================

df["streams"] = pd.to_numeric(
    df["streams"],
    errors="coerce"
)

for feature in audio_features:

    df[feature] = pd.to_numeric(
        df[feature],
        errors="coerce"
    )


# ============================================================
# KEEP ONLY DATA REQUIRED FOR THIS QUESTION
# ============================================================

df_q1 = df.dropna(
    subset=["streams"] + audio_features
).copy()


# ============================================================
# REMOVE INVALID STREAM VALUES
# ============================================================

df_q1 = df_q1[
    df_q1["streams"] > 0
].copy()


# ============================================================
# LOG STREAMS
# ============================================================

df_q1["log_streams"] = np.log10(
    df_q1["streams"]
)


# ============================================================
# FIND STRONGEST AUDIO FEATURES
# ============================================================

correlations = (
    df_q1[audio_features + ["streams"]]
    .corr()["streams"]
    .drop("streams")
    .sort_values(
        key=abs,
        ascending=False
    )
)

top_feature_1 = correlations.index[0]
top_feature_2 = correlations.index[1]

corr_1 = correlations.iloc[0]
corr_2 = correlations.iloc[1]


# ============================================================
# HEADER METRICS
# ============================================================

col1, col2, col3 = st.columns(3)

with col1:
    st.metric(
        "🥇 Strongest Feature",
        top_feature_1.title(),
        f"r = {corr_1:.3f}"
    )

with col2:
    st.metric(
        "🥈 Second Strongest",
        top_feature_2.title(),
        f"r = {corr_2:.3f}"
    )

with col3:
    st.metric(
        "🎵 Tracks Analysed",
        f"{len(df_q1):,}"
    )


# ============================================================
# FEATURE CONTROLS
# ============================================================

st.sidebar.header("🎛️ Explore the Spotify Data")

st.sidebar.markdown(
    """
    **How to use the graph**

    🖱️ Drag → Rotate the 3D view
    🔍 Scroll → Zoom
    👆 Hover → Inspect a song
    """
)

# Allow the user to change X and Y features
x_feature = st.sidebar.selectbox(
    "X-axis audio feature",
    audio_features,
    index=audio_features.index(top_feature_1)
)

y_feature = st.sidebar.selectbox(
    "Y-axis audio feature",
    audio_features,
    index=audio_features.index(top_feature_2)
)


# ============================================================
# BUILD 3D GRAPH
# ============================================================

fig = go.Figure()


fig.add_trace(
    go.Scatter3d(

        x=df_q1[x_feature],

        y=df_q1[y_feature],

        z=df_q1["log_streams"],

        mode="markers",

        marker=dict(

            size=6,

            color=df_q1["log_streams"],

            colorscale="Turbo",

            opacity=0.75,

            colorbar=dict(
                title="Log₁₀ Streams"
            )
        ),

        text=df_q1["track_name"],

        customdata=np.column_stack([
            df_q1["artist_name"],
            df_q1["streams"],
            df_q1["danceability"],
            df_q1["energy"],
            df_q1["speechiness"],
            df_q1["valence"]
        ]),

        hovertemplate=(

            "<b>%{text}</b><br><br>"

            "🎤 Artist: %{customdata[0]}<br>"

            "▶️ Streams: %{customdata[1]:,.0f}<br><br>"

            + x_feature.title() +
            ": %{x:.1f}<br>"

            + y_feature.title() +
            ": %{y:.1f}<br><br>"

            "Danceability: %{customdata[2]:.1f}%<br>"

            "Energy: %{customdata[3]:.1f}%<br>"

            "Speechiness: %{customdata[4]:.1f}%<br>"

            "Valence: %{customdata[5]:.1f}%"

            "<extra></extra>"
        ),

        name="Tracks"
    )
)


# ============================================================
# GRAPH DESIGN
# ============================================================

fig.update_layout(

    title=dict(

        text=(
            f"🎵 Spotify Hit DNA: "
            f"{x_feature.title()} × "
            f"{y_feature.title()} × Streams"
        ),

        font=dict(
            size=26
        ),

        x=0.5
    ),

    scene=dict(

        xaxis=dict(
            title=f"{x_feature.title()} (%)",
            backgroundcolor="rgba(0,0,0,0)"
        ),

        yaxis=dict(
            title=f"{y_feature.title()} (%)",
            backgroundcolor="rgba(0,0,0,0)"
        ),

        zaxis=dict(
            title="Spotify Streams — Log Scale",
            backgroundcolor="rgba(0,0,0,0)"
        ),

        camera=dict(
            eye=dict(
                x=1.6,
                y=1.6,
                z=1.25
            )
        )
    ),

    height=750,

    margin=dict(
        l=0,
        r=0,
        t=80,
        b=0
    ),

    paper_bgcolor="rgba(0,0,0,0)",

    plot_bgcolor="rgba(0,0,0,0)"
)


# ============================================================
# DISPLAY GRAPH
# ============================================================

st.plotly_chart(
    fig,
    use_container_width=True
)


# ============================================================
# INSIGHT
# ============================================================

st.subheader("💡 What does the data tell us?")

st.markdown(
    f"""
    **{top_feature_1.title()}** has the strongest linear association
    with Spotify streams in this dataset (**r = {corr_1:.3f}**),
    followed by **{top_feature_2.title()}**
    (**r = {corr_2:.3f}**).

    However, both correlations are relatively weak. This suggests that
    **audio characteristics alone do not strongly explain streaming
    success**.
    """
)

st.caption(
    "Note: Correlation indicates association, not causation. "
    "The 3D height uses log₁₀(streams) so extremely large stream counts "
    "remain visually interpretable."
)

Writing app.py


In [25]:
!streamlit run app.py &>/content/streamlit.log &

In [27]:
!streamlit run app.py --server.port 8501 > /content/streamlit.log 2>&1 &

In [28]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8501)")
print("🔥 YOUR STREAMLIT APP:")
print(url)

🔥 YOUR STREAMLIT APP:
https://8501-m-s-kkb-usc1b1-21hel5vn7zyp4-b.us-central1-1.prod.colab.dev


In [30]:

!pkill -f streamlit || true

^C


In [31]:
!streamlit run app.py --server.address 0.0.0.0 --server.port 8501 > /content/streamlit.log 2>&1 &

In [32]:
!cat /content/streamlit.log




2026-09-14 07:16:46.774 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.70.121.115:8501

2026-09-14 07:16:56.678 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-kkb-usc1b1-21hel5vn7zyp4-b.us-central1-1.prod.colab.dev, host=m-s-kkb-usc1b1-21hel5vn7zyp4.us-central1-b.c.codatalab-user-runtimes.internal:8007
2026-09-14 07:16:58.029 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-kkb-usc1b1-21hel5vn7zyp4-b.us-central1-1.prod.colab.dev, host=m-s-kkb-usc1b1-21hel5vn7zyp4.us-central1-b.c.codatalab-user-runtimes.internal:8007
2026-09-14 07:16:59.650 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-kkb-usc1b1-21hel5vn7zyp4-b.us-central1-1.prod.colab.dev, host=m-s-kkb-usc1b1-21hel5vn7zyp4.us-central1-b.c.codatalab-

In [33]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Mon, 14 Sep 2026 07:18:25 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7459
last-modified: Mon, 14 Sep 2026 06:51:37 GMT
etag: "cbbfe301ca782e74ffed1e90134ffd01"
cache-control: no-cache



In [34]:
from google.colab import output
output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [35]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Mon, 14 Sep 2026 07:20:03 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7459
last-modified: Mon, 14 Sep 2026 06:51:37 GMT
etag: "cbbfe301ca782e74ffed1e90134ffd01"
cache-control: no-cache



In [36]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501, height=900)

<IPython.core.display.Javascript object>

In [38]:
!streamlit run app.py \
    --server.address 0.0.0.0 \
    --server.port 8501 \
    --server.headless true \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    > /content/streamlit.log 2>&1 &

In [39]:
!cat /content/streamlit.log



2026-09-14 07:24:41.668 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.70.121.115:8501

2026-09-14 07:24:51.947 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


In [40]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Spotify Hit DNA",
    page_icon="🎵",
    layout="wide"
)


# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown("""
<style>

.main {
    background-color: #080b12;
}

.block-container {
    padding-top: 2rem;
    padding-bottom: 2rem;
}

h1 {
    font-size: 42px !important;
    font-weight: 800 !important;
}

h2 {
    font-size: 28px !important;
    font-weight: 700 !important;
}

[data-testid="stMetricValue"] {
    font-size: 30px;
    font-weight: 800;
}

</style>
""", unsafe_allow_html=True)


# ============================================================
# LOAD DATA
# ============================================================

file_path = "/content/spotify dataset final.xlsm"

df = pd.read_excel(file_path)


# ============================================================
# RENAME COLUMNS
# ============================================================

df.columns = [
    "track_name",
    "artist_name",
    "artist_count",
    "released_year",
    "released_month",
    "released_day",
    "in_spotify_playlists",
    "in_spotify_charts",
    "streams",
    "in_apple_playlists",
    "in_apple_charts",
    "in_deezer_playlists",
    "in_deezer_charts",
    "in_shazam_charts",
    "bpm",
    "key",
    "mode",
    "danceability",
    "valence",
    "energy",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "cover_url"
]


# ============================================================
# CLEAN REQUIRED DATA
# ============================================================

numeric_columns = [
    "released_year",
    "streams",
    "bpm",
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness"
]

for column in numeric_columns:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# Remove unusable values

df = df.dropna(
    subset=[
        "released_year",
        "streams",
        "bpm"
    ]
).copy()


df = df[
    (df["streams"] > 0) &
    (df["bpm"] > 0)
].copy()


df["released_year"] = df["released_year"].astype(int)


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.title("🎛️ Spotify Hit Explorer")

question = st.sidebar.radio(
    "Choose your question",
    [
        "Question 1 — Hit DNA",
        "Question 2 — Tempo Evolution"
    ]
)


# ============================================================
# ============================================================
# QUESTION 1
# ============================================================
# ============================================================

if question == "Question 1 — Hit DNA":

    st.title("🎵 Spotify Hit DNA")

    st.markdown("""
    ### What audio features most strongly correlate with high stream counts?

    Rotate the 3D visualization, zoom, and hover over individual tracks
    to explore the relationship between audio characteristics and streams.
    """)


    # --------------------------------------------------------
    # Q1 AUDIO FEATURES
    # --------------------------------------------------------

    audio_features = [
        "danceability",
        "energy",
        "valence",
        "acousticness",
        "instrumentalness",
        "liveness",
        "speechiness",
        "bpm"
    ]


    df_q1 = df.dropna(
        subset=["streams"] + audio_features
    ).copy()


    df_q1["log_streams"] = np.log10(
        df_q1["streams"]
    )


    # --------------------------------------------------------
    # CORRELATIONS
    # --------------------------------------------------------

    correlations = (
        df_q1[
            audio_features + ["streams"]
        ]
        .corr()["streams"]
        .drop("streams")
        .sort_values(
            key=abs,
            ascending=False
        )
    )


    top_feature_1 = correlations.index[0]
    top_feature_2 = correlations.index[1]

    corr_1 = correlations.iloc[0]
    corr_2 = correlations.iloc[1]


    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    col1, col2, col3 = st.columns(3)

    with col1:

        st.metric(
            "🥇 Strongest Feature",
            top_feature_1.title(),
            f"r = {corr_1:.3f}"
        )

    with col2:

        st.metric(
            "🥈 Second Strongest",
            top_feature_2.title(),
            f"r = {corr_2:.3f}"
        )

    with col3:

        st.metric(
            "🎵 Tracks Analysed",
            f"{len(df_q1):,}"
        )


    # --------------------------------------------------------
    # Q1 FEATURE CONTROLS
    # --------------------------------------------------------

    st.sidebar.markdown("---")

    st.sidebar.subheader("Q1 — 3D Controls")


    x_feature = st.sidebar.selectbox(
        "X-axis",
        audio_features,
        index=audio_features.index(
            top_feature_1
        )
    )


    y_feature = st.sidebar.selectbox(
        "Y-axis",
        audio_features,
        index=audio_features.index(
            top_feature_2
        )
    )


    # --------------------------------------------------------
    # Q1 3D GRAPH
    # --------------------------------------------------------

    fig = go.Figure()


    fig.add_trace(
        go.Scatter3d(

            x=df_q1[x_feature],

            y=df_q1[y_feature],

            z=df_q1["log_streams"],

            mode="markers",

            marker=dict(
                size=6,
                color=df_q1["log_streams"],
                colorscale="Turbo",
                opacity=0.78,

                colorbar=dict(
                    title="Log₁₀<br>Streams"
                )
            ),

            text=df_q1["track_name"],

            customdata=np.column_stack([
                df_q1["artist_name"],
                df_q1["streams"],
                df_q1["danceability"],
                df_q1["energy"],
                df_q1["speechiness"],
                df_q1["valence"]
            ]),

            hovertemplate=(

                "<b>%{text}</b><br><br>"

                "🎤 Artist: %{customdata[0]}<br>"

                "▶️ Streams: %{customdata[1]:,.0f}<br><br>"

                + x_feature.title() +
                ": %{x:.1f}%<br>"

                + y_feature.title() +
                ": %{y:.1f}%<br><br>"

                "Danceability: %{customdata[2]:.1f}%<br>"

                "Energy: %{customdata[3]:.1f}%<br>"

                "Speechiness: %{customdata[4]:.1f}%<br>"

                "Valence: %{customdata[5]:.1f}%"

                "<extra></extra>"
            )
        )
    )


    fig.update_layout(

        title=dict(
            text=(
                f"🎵 Spotify Hit DNA: "
                f"{x_feature.title()} × "
                f"{y_feature.title()} × Streams"
            ),
            x=0.5,
            font=dict(size=27)
        ),

        scene=dict(

            xaxis=dict(
                title=f"{x_feature.title()} (%)"
            ),

            yaxis=dict(
                title=f"{y_feature.title()} (%)"
            ),

            zaxis=dict(
                title="Spotify Streams — Log Scale"
            ),

            camera=dict(
                eye=dict(
                    x=1.6,
                    y=1.6,
                    z=1.25
                )
            )
        ),

        height=750,

        margin=dict(
            l=0,
            r=0,
            t=90,
            b=0
        ),

        template="plotly_dark"
    )


    st.plotly_chart(
        fig,
        width="stretch",
        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "displaylogo": False
        }
    )


    st.subheader("💡 Q1 Finding")

    st.markdown(
        f"""
        **{top_feature_1.title()}** has the strongest linear association
        with Spotify streams in this dataset, with a correlation of
        **{corr_1:.3f}**.

        The relationship is relatively weak, which suggests that
        audio characteristics alone do not strongly explain streaming
        success.
        """
    )

    st.caption(
        "Correlation indicates association, not causation."
    )


# ============================================================
# ============================================================
# QUESTION 2
# ============================================================
# ============================================================

else:

    st.title("🔥 The Tempo of a Hit")

    st.markdown("""
    ### How has the average tempo of top hits changed over recent years?

    **Top hits are defined as the highest-streamed tracks within each year.**

    Rotate the 3D world, zoom into specific years, and hover over individual
    tracks to discover how the tempo of popular music has evolved.
    """)


    # ========================================================
    # Q2 SETTINGS
    # ========================================================

    st.sidebar.markdown("---")

    st.sidebar.subheader("Q2 — Tempo Controls")


    # --------------------------------------------------------
    # AVAILABLE YEARS
    # --------------------------------------------------------

    available_years = sorted(
        df["released_year"].unique()
    )


    latest_year = max(
        available_years
    )


    # Default = latest 10 years available

    default_start = max(
        min(available_years),
        latest_year - 9
    )


    year_range = st.sidebar.slider(

        "📅 Recent years",

        min_value=min(
            available_years
        ),

        max_value=max(
            available_years
        ),

        value=(
            default_start,
            latest_year
        ),

        step=1
    )


    # --------------------------------------------------------
    # TOP HIT DEFINITION
    # --------------------------------------------------------

    top_percent = st.sidebar.select_slider(

        "🔥 Define a Top Hit",

        options=[
            5,
            10,
            15,
            20,
            25
        ],

        value=10,

        format_func=lambda x:
            f"Top {x}% streamed tracks"
    )


    # ========================================================
    # FILTER RECENT YEARS
    # ========================================================

    df_q2 = df[
        (df["released_year"] >= year_range[0]) &
        (df["released_year"] <= year_range[1])
    ].copy()


    # ========================================================
    # IDENTIFY TOP HITS
    # ========================================================

    # Rank songs separately inside every year.
    #
    # Example:
    # Top 10% = songs in the highest 10% of streams
    # within that particular year.

    df_q2["stream_percentile"] = (

        df_q2
        .groupby("released_year")["streams"]
        .rank(
            pct=True,
            ascending=True
        )
    )


    df_q2["is_top_hit"] = (
        df_q2["stream_percentile"]
        >= (1 - top_percent / 100)
    )


    top_hits = df_q2[
        df_q2["is_top_hit"]
    ].copy()


    # ========================================================
    # YEARLY SUMMARY
    # ========================================================

    yearly = (

        top_hits
        .groupby("released_year")
        .agg(

            average_bpm=(
                "bpm",
                "mean"
            ),

            median_bpm=(
                "bpm",
                "median"
            ),

            tracks=(
                "track_name",
                "count"
            ),

            average_streams=(
                "streams",
                "mean"
            )

        )

        .reset_index()

        .sort_values(
            "released_year"
        )
    )


    # Log popularity for 3D visualization

    top_hits["log_streams"] = np.log10(
        top_hits["streams"]
    )


    yearly["log_average_streams"] = np.log10(
        yearly["average_streams"]
    )


    # ========================================================
    # 3-YEAR ROLLING TEMPO
    # ========================================================

    yearly["rolling_bpm"] = (

        yearly["average_bpm"]

        .rolling(
            window=3,
            min_periods=1
        )

        .mean()
    )


    # ========================================================
    # CALCULATE OVERALL CHANGE
    # ========================================================

    if len(yearly) >= 2:

        first_year = yearly.iloc[0]

        last_year = yearly.iloc[-1]

        first_bpm = first_year["average_bpm"]

        last_bpm = last_year["average_bpm"]

        bpm_change = (
            last_bpm - first_bpm
        )

        percentage_change = (
            bpm_change / first_bpm
        ) * 100

    else:

        first_bpm = yearly.iloc[0]["average_bpm"]

        last_bpm = first_bpm

        bpm_change = 0

        percentage_change = 0


    # ========================================================
    # TOP METRICS
    # ========================================================

    col1, col2, col3, col4 = st.columns(4)


    with col1:

        st.metric(
            "📅 First Year",
            str(
                int(
                    yearly.iloc[0]["released_year"]
                )
            )
        )


    with col2:

        st.metric(
            "🎵 First-Year Avg BPM",
            f"{first_bpm:.1f}"
        )


    with col3:

        st.metric(
            "🔥 Latest-Year Avg BPM",
            f"{last_bpm:.1f}",
            f"{bpm_change:+.1f} BPM"
        )


    with col4:

        st.metric(
            "🎧 Top Hits",
            f"{len(top_hits):,}"
        )


    # ========================================================
    # THE 3D VISUALIZATION
    # ========================================================

    fig = go.Figure()


    # ========================================================
    # LAYER 1 — INDIVIDUAL TOP HITS
    # ========================================================

    fig.add_trace(

        go.Scatter3d(

            x=top_hits["released_year"],

            y=top_hits["bpm"],

            z=top_hits["log_streams"],

            mode="markers",

            name="Top Hit Tracks",

            marker=dict(

                size=7,

                color=top_hits["bpm"],

                colorscale="Turbo",

                opacity=0.78,

                colorbar=dict(
                    title="BPM"
                )
            ),

            text=top_hits["track_name"],

            customdata=np.column_stack([

                top_hits["artist_name"],

                top_hits["streams"],

                top_hits["released_year"],

                top_hits["bpm"]

            ]),

            hovertemplate=(

                "<b>🎵 %{text}</b><br><br>"

                "🎤 Artist: %{customdata[0]}<br>"

                "📅 Year: %{customdata[2]:.0f}<br>"

                "🥁 Tempo: <b>%{customdata[3]:.1f} BPM</b><br>"

                "▶️ Streams: %{customdata[1]:,.0f}<br><br>"

                "Height = Log₁₀(Streams)<br>"

                "<extra>Top Hit</extra>"
            )
        )
    )


    # ========================================================
    # LAYER 2 — YEARLY AVERAGE TEMPO
    # ========================================================

    fig.add_trace(

        go.Scatter3d(

            x=yearly["released_year"],

            y=yearly["average_bpm"],

            z=yearly["log_average_streams"],

            mode="lines+markers",

            name="Average BPM",

            line=dict(

                width=10

            ),

            marker=dict(

                size=13,

                color=yearly["average_bpm"],

                colorscale="Turbo",

                showscale=False

            ),

            customdata=np.column_stack([

                yearly["tracks"],

                yearly["median_bpm"],

                yearly["rolling_bpm"],

                yearly["average_streams"]

            ]),

            hovertemplate=(

                "<b>📅 %{x:.0f}</b><br><br>"

                "🔥 Average Tempo: "
                "<b>%{y:.1f} BPM</b><br>"

                "Median Tempo: %{customdata[1]:.1f} BPM<br>"

                "Top Hits: %{customdata[0]:.0f}<br>"

                "3-Year Rolling BPM: "
                "%{customdata[2]:.1f}<br>"

                "Average Streams: "
                "%{customdata[3]:,.0f}"

                "<extra>YEARLY AVERAGE</extra>"
            )
        )
    )


    # ========================================================
    # LAYOUT
    # ========================================================

    fig.update_layout(

        title=dict(

            text=(
                "🔥 The Evolution of Hit Tempo"
                "<br>"
                f"<sup>"
                f"Top {top_percent}% streamed tracks • "
                f"{year_range[0]}–{year_range[1]}"
                f"</sup>"
            ),

            x=0.5,

            font=dict(
                size=29
            )
        ),


        scene=dict(

            xaxis=dict(

                title="📅 Release Year",

                dtick=1,

                gridcolor="rgba(255,255,255,0.12)"
            ),


            yaxis=dict(

                title="🥁 Tempo (BPM)",

                gridcolor="rgba(255,255,255,0.12)"
            ),


            zaxis=dict(

                title="🔥 Popularity — Log₁₀ Streams",

                gridcolor="rgba(255,255,255,0.12)"
            ),


            camera=dict(

                eye=dict(

                    x=1.65,

                    y=1.65,

                    z=1.35
                )
            )
        ),


        height=800,


        margin=dict(

            l=0,

            r=0,

            t=110,

            b=0
        ),


        legend=dict(

            orientation="h",

            y=1.02,

            x=0.5,

            xanchor="center"
        ),


        template="plotly_dark"
    )


    # ========================================================
    # DISPLAY
    # ========================================================

    st.plotly_chart(

        fig,

        width="stretch",

        config={

            "displayModeBar": True,

            "scrollZoom": True,

            "displaylogo": False,

            "modeBarButtonsToAdd": [

                "resetCameraDefault3d",

                "resetCameraLastSave3d"

            ]
        }
    )


    # ========================================================
    # INSIGHT
    # ========================================================

    st.subheader("💡 What does the data tell us?")


    if bpm_change > 0:

        st.success(

            f"""
            ### 🚀 Hit tempo increased

            The average tempo of the top {top_percent}% streamed tracks
            changed from **{first_bpm:.1f} BPM** in
            **{int(yearly.iloc[0]["released_year"])}**
            to **{last_bpm:.1f} BPM** in
            **{int(yearly.iloc[-1]["released_year"])}**.

            That's an overall increase of
            **{bpm_change:+.1f} BPM ({percentage_change:+.1f}%)**.
            """
        )


    elif bpm_change < 0:

        st.info(

            f"""
            ### 🎧 Hit tempo decreased

            The average tempo of the top {top_percent}% streamed tracks
            changed from **{first_bpm:.1f} BPM** in
            **{int(yearly.iloc[0]["released_year"])}**
            to **{last_bpm:.1f} BPM** in
            **{int(yearly.iloc[-1]["released_year"])}**.

            That's an overall decrease of
            **{abs(bpm_change):.1f} BPM ({abs(percentage_change):.1f}%)**.
            """
        )


    else:

        st.warning(

            """
            ### ⚖️ Tempo remained relatively stable

            The average BPM of top hits remained almost unchanged
            across the selected period.
            """
        )


    # ========================================================
    # EXPLANATION FOR JUDGES
    # ========================================================

    with st.expander("🔬 How was this calculated?"):

        st.markdown(

            f"""
            **Definition of a top hit:**
            The top **{top_percent}% of streamed tracks within each year**.

            **Average tempo:**
            The arithmetic mean of the BPM values of those top hits.

            **3D visualization:**

            - **X-axis:** Release year
            - **Y-axis:** BPM
            - **Z-axis:** Logarithm of stream count
            - **Small points:** Individual top-hit tracks
            - **Large connected points:** Yearly average BPM

            The logarithmic stream scale prevents extremely large
            stream counts from dominating the visualization.
            """
        )


    st.caption(
        "Tempo is measured in beats per minute (BPM). "
        "The analysis shows changes in association over time and "
        "does not imply that tempo causes streaming success."
    )

Overwriting app.py


In [41]:
!pkill -f streamlit || true


^C


In [42]:
!streamlit run app.py \
    --server.address 0.0.0.0 \
    --server.port 8501 \
    --server.headless true \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    > /content/streamlit.log 2>&1 &

In [43]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Mon, 14 Sep 2026 07:41:17 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7459
last-modified: Mon, 14 Sep 2026 06:51:37 GMT
etag: "cbbfe301ca782e74ffed1e90134ffd01"
cache-control: no-cache



In [44]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501, height=900)

<IPython.core.display.Javascript object>

In [45]:
!ls

 app.py       'spotify dataset final (1).xlsm'	 streamlit.log
 sample_data  'spotify dataset final.xlsm'


In [51]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Spotify Analytics — Hackathon",
    page_icon="🎵",
    layout="wide"
)


# ============================================================
# HEADER
# ============================================================

st.title("🎵 Spotify Analytics — Interactive Hit Lab")

st.markdown(
    """
    ### Explore what makes Spotify hits successful
    Three interactive 3D investigations into **streaming success, tempo, and collaborations**.
    """
)


# ============================================================
# LOAD DATA
# ============================================================

FILE_PATH = "/content/spotify dataset final.xlsm"

df = pd.read_excel(FILE_PATH)


# ============================================================
# RENAME COLUMNS
# ============================================================

df.columns = [
    "track_name",
    "artist_name",
    "artist_count",
    "released_year",
    "released_month",
    "released_day",
    "in_spotify_playlists",
    "in_spotify_charts",
    "streams",
    "in_apple_playlists",
    "in_apple_charts",
    "in_deezer_playlists",
    "in_deezer_charts",
    "in_shazam_charts",
    "bpm",
    "key",
    "mode",
    "danceability",
    "valence",
    "energy",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "cover_url"
]


# ============================================================
# NUMERIC CLEANING
# ============================================================

numeric_columns = [
    "artist_count",
    "released_year",
    "released_month",
    "released_day",
    "in_spotify_playlists",
    "in_spotify_charts",
    "streams",
    "in_apple_playlists",
    "in_apple_charts",
    "in_deezer_playlists",
    "in_deezer_charts",
    "in_shazam_charts",
    "bpm",
    "danceability",
    "valence",
    "energy",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# ============================================================
# SIDEBAR NAVIGATION
# ============================================================

st.sidebar.title("🎛️ Spotify Hit Lab")

question = st.sidebar.radio(
    "Choose an investigation:",
    [
        "Question 1 — Hit DNA",
        "Question 2 — Tempo Evolution",
        "Question 3 — Solo vs Collaboration"
    ]
)


# ============================================================
# COMMON YEAR FILTER
# ============================================================

available_years = sorted(
    df["released_year"].dropna().astype(int).unique()
)

min_year = min(available_years)
max_year = max(available_years)

default_start = max(min_year, max_year - 9)

year_range = st.sidebar.slider(
    "📅 Release year range",
    min_value=min_year,
    max_value=max_year,
    value=(default_start, max_year),
    step=1
)


# ============================================================
# QUESTION 1
# ============================================================

if question == "Question 1 — Hit DNA":

    st.header("🧬 Question 1 — Spotify Hit DNA")

    st.markdown(
        """
        **What audio features most strongly correlate with high stream counts?**
        """
    )

    audio_features = [
        "danceability",
        "energy",
        "valence",
        "acousticness",
        "instrumentalness",
        "liveness",
        "speechiness",
        "bpm"
    ]

    df_q1 = df.dropna(
        subset=["streams"] + audio_features
    ).copy()

    df_q1 = df_q1[
        (df_q1["streams"] > 0) &
        (df_q1["released_year"] >= year_range[0]) &
        (df_q1["released_year"] <= year_range[1])
    ]

    df_q1["log_streams"] = np.log10(df_q1["streams"])

    correlations = (
        df_q1[audio_features + ["streams"]]
        .corr()["streams"]
        .drop("streams")
        .sort_values(key=lambda x: abs(x), ascending=False)
    )

    top_feature_1 = correlations.index[0]
    top_feature_2 = correlations.index[1]

    col1, col2, col3 = st.columns(3)

    with col1:
        st.metric(
            "🥇 Strongest relationship",
            top_feature_1.title(),
            f"{correlations[top_feature_1]:.3f}"
        )

    with col2:
        st.metric(
            "🥈 Second strongest",
            top_feature_2.title(),
            f"{correlations[top_feature_2]:.3f}"
        )

    with col3:
        st.metric(
            "🎵 Tracks analyzed",
            f"{len(df_q1):,}"
        )

    st.markdown("### 🌌 Rotate the 3D Spotify Hit Universe")

    feature_x = st.sidebar.selectbox(
        "X-axis audio feature",
        audio_features,
        index=audio_features.index(top_feature_1)
    )

    feature_y = st.sidebar.selectbox(
        "Y-axis audio feature",
        audio_features,
        index=audio_features.index(top_feature_2)
    )

    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=df_q1[feature_x],
            y=df_q1[feature_y],
            z=df_q1["log_streams"],

            mode="markers",

            marker=dict(
                size=7,
                color=df_q1["log_streams"],
                colorscale="Turbo",
                opacity=0.78,
                colorbar=dict(
                    title="Log₁₀ Streams"
                )
            ),

            text=df_q1["track_name"],

            customdata=np.column_stack([
                df_q1["artist_name"],
                df_q1["streams"],
                df_q1["released_year"],
                df_q1[feature_x],
                df_q1[feature_y]
            ]),

            hovertemplate=
                "<b>🎵 %{text}</b><br><br>"
                "🎤 Artist: %{customdata[0]}<br>"
                "📅 Year: %{customdata[2]:.0f}<br>"
                f"🎚️ {feature_x.title()}: <b>%{{customdata[3]:.1f}}</b><br>"
                f"🎚️ {feature_y.title()}: <b>%{{customdata[4]:.1f}}</b><br>"
                "▶️ Streams: <b>%{customdata[1]:,.0f}</b>"
                "<extra></extra>"
        )
    )

    fig.update_layout(
        title=f"Spotify Hit DNA — {feature_x.title()} × {feature_y.title()} × Streams",

        template="plotly_dark",

        height=800,

        scene=dict(
            xaxis_title=feature_x.title(),
            yaxis_title=feature_y.title(),
            zaxis_title="Log₁₀(Streams)",

            camera=dict(
                eye=dict(
                    x=1.6,
                    y=1.6,
                    z=1.3
                )
            )
        ),

        margin=dict(
            l=0,
            r=0,
            b=0,
            t=60
        )
    )

    st.plotly_chart(
        fig,
        width="stretch",
        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "displaylogo": False,
            "modeBarButtonsToAdd": [
                "resetCameraDefault3d",
                "resetCameraLastSave3d"
            ]
        }
    )

    with st.expander("🧠 What does this mean?"):

        st.write(
            f"""
            **{top_feature_1.title()}** has the strongest Pearson correlation
            with streams in this dataset at **{correlations[top_feature_1]:.3f}**.

            However, the relationship is weak, meaning audio features alone
            do not strongly explain streaming success.

            **Important:** correlation does not imply causation.
            """
        )


# ============================================================
# QUESTION 2
# ============================================================

elif question == "Question 2 — Tempo Evolution":

    st.header("🥁 Question 2 — Tempo Evolution")

    st.markdown(
        """
        **How has the average tempo of top hits changed over recent years?**
        """
    )

    top_percent = st.sidebar.select_slider(
        "🔥 Define a Top Hit",
        options=[5, 10, 15, 20, 25],
        value=10,
        format_func=lambda x: f"Top {x}% streamed tracks"
    )

    df_q2 = df[
        (df["released_year"] >= year_range[0]) &
        (df["released_year"] <= year_range[1])
    ].copy()

    df_q2 = df_q2.dropna(
        subset=["released_year", "streams", "bpm"]
    )

    df_q2 = df_q2[df_q2["streams"] > 0]
    df_q2 = df_q2[df_q2["bpm"] > 0]

    df_q2["stream_percentile"] = (
        df_q2
        .groupby("released_year")["streams"]
        .rank(
            pct=True,
            ascending=True
        )
    )

    df_q2["is_top_hit"] = (
        df_q2["stream_percentile"]
        >= (1 - top_percent / 100)
    )

    top_hits = df_q2[
        df_q2["is_top_hit"]
    ].copy()

    yearly = (
        top_hits
        .groupby("released_year")
        .agg(
            average_bpm=("bpm", "mean"),
            median_bpm=("bpm", "median"),
            tracks=("track_name", "count"),
            average_streams=("streams", "mean")
        )
        .reset_index()
        .sort_values("released_year")
    )

    top_hits["log_streams"] = np.log10(
        top_hits["streams"]
    )

    yearly["log_average_streams"] = np.log10(
        yearly["average_streams"]
    )

    yearly["rolling_bpm"] = (
        yearly["average_bpm"]
        .rolling(
            window=3,
            min_periods=1
        )
        .mean()
    )

    first_year = yearly.iloc[0]
    latest_year = yearly.iloc[-1]

    bpm_change = (
        latest_year["average_bpm"]
        - first_year["average_bpm"]
    )

    col1, col2, col3, col4 = st.columns(4)

    with col1:
        st.metric(
            "📅 First year",
            f"{int(first_year['released_year'])}"
        )

    with col2:
        st.metric(
            "🥁 First average BPM",
            f"{first_year['average_bpm']:.1f}"
        )

    with col3:
        st.metric(
            "🚀 Latest average BPM",
            f"{latest_year['average_bpm']:.1f}"
        )

    with col4:
        st.metric(
            "📈 BPM change",
            f"{bpm_change:+.1f}"
        )

    st.markdown(
        "### 🌌 Spin through the evolution of Spotify's top hits"
    )

    fig = go.Figure()

    # Individual tracks
    fig.add_trace(
        go.Scatter3d(
            x=top_hits["released_year"],
            y=top_hits["bpm"],
            z=top_hits["log_streams"],

            mode="markers",

            name="Top Hit Tracks",

            marker=dict(
                size=7,
                color=top_hits["bpm"],
                colorscale="Turbo",
                opacity=0.78,
                colorbar=dict(
                    title="BPM"
                )
            ),

            text=top_hits["track_name"],

            customdata=np.column_stack([
                top_hits["artist_name"],
                top_hits["streams"],
                top_hits["released_year"],
                top_hits["bpm"]
            ]),

            hovertemplate=
                "<b>🎵 %{text}</b><br><br>"
                "🎤 Artist: %{customdata[0]}<br>"
                "📅 Year: %{customdata[2]:.0f}<br>"
                "🥁 Tempo: <b>%{customdata[3]:.1f} BPM</b><br>"
                "▶️ Streams: <b>%{customdata[1]:,.0f}</b>"
                "<extra>TOP HIT</extra>"
        )
    )

    # Yearly average
    fig.add_trace(
        go.Scatter3d(
            x=yearly["released_year"],
            y=yearly["average_bpm"],
            z=yearly["log_average_streams"],

            mode="lines+markers",

            name="Average BPM",

            line=dict(
                width=10
            ),

            marker=dict(
                size=13,
                color=yearly["average_bpm"],
                colorscale="Turbo",
                showscale=False
            ),

            customdata=np.column_stack([
                yearly["tracks"],
                yearly["median_bpm"],
                yearly["rolling_bpm"],
                yearly["average_streams"]
            ]),

            hovertemplate=
                "<b>📅 %{x:.0f}</b><br><br>"
                "🔥 Average Tempo: <b>%{y:.1f} BPM</b><br>"
                "Median Tempo: %{customdata[1]:.1f} BPM<br>"
                "Top Hits: %{customdata[0]:.0f}<br>"
                "3-Year Rolling BPM: %{customdata[2]:.1f}<br>"
                "Average Streams: %{customdata[3]:,.0f}"
                "<extra>YEARLY AVERAGE</extra>"
        )
    )

    fig.update_layout(
        title="Spotify Top-Hit Tempo Evolution",

        template="plotly_dark",

        height=800,

        scene=dict(
            xaxis_title="Release Year",
            yaxis_title="BPM",
            zaxis_title="Log₁₀(Streams)",

            camera=dict(
                eye=dict(
                    x=1.6,
                    y=1.6,
                    z=1.3
                )
            )
        ),

        margin=dict(
            l=0,
            r=0,
            b=0,
            t=60
        )
    )

    st.plotly_chart(
        fig,
        width="stretch",
        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "displaylogo": False,
            "modeBarButtonsToAdd": [
                "resetCameraDefault3d",
                "resetCameraLastSave3d"
            ]
        }
    )

    if bpm_change > 2:
        insight = "📈 Top-hit tempo has generally increased."
    elif bpm_change < -2:
        insight = "📉 Top-hit tempo has generally decreased."
    else:
        insight = "➡️ Top-hit tempo has remained relatively stable."

    st.success(insight)

    with st.expander("🧠 Methodology"):
        st.write(
            f"""
            A **Top Hit** is defined as a track belonging to the highest
            **{top_percent}% of streamed tracks within its release year**.

            This prevents older years from automatically dominating the
            comparison simply because their tracks have had more time to
            accumulate streams.

            The connected line represents the yearly average BPM.
            """
        )


# ============================================================
# QUESTION 3
# ============================================================

else:

    st.header("🤝 Question 3 — Solo vs Collaboration")

    st.markdown(
        """
        **Do solo artists or collaborations tend to have higher streaming success?**
        """
    )

    # --------------------------------------------------------
    # FILTER DATA
    # --------------------------------------------------------

    df_q3 = df[
        (df["released_year"] >= year_range[0]) &
        (df["released_year"] <= year_range[1])
    ].copy()

    df_q3 = df_q3.dropna(
        subset=[
            "streams",
            "artist_count",
            "released_year"
        ]
    )

    df_q3 = df_q3[
        df_q3["streams"] > 0
    ]

    df_q3 = df_q3[
        df_q3["artist_count"] >= 1
    ]

    # --------------------------------------------------------
    # CLASSIFY SOLO / COLLABORATION
    # --------------------------------------------------------

    df_q3["artist_type"] = np.where(
        df_q3["artist_count"] == 1,
        "Solo",
        "Collaboration"
    )

    df_q3["log_streams"] = np.log10(
        df_q3["streams"]
    )

    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    summary = (
        df_q3
        .groupby("artist_type")
        .agg(
            average_streams=("streams", "mean"),
            median_streams=("streams", "median"),
            average_artist_count=("artist_count", "mean"),
            tracks=("track_name", "count"),
            average_log_streams=("log_streams", "mean")
        )
        .reset_index()
    )

    solo = summary[
        summary["artist_type"] == "Solo"
    ]

    collaboration = summary[
        summary["artist_type"] == "Collaboration"
    ]

    solo_avg = (
        solo["average_streams"].iloc[0]
        if len(solo) > 0 else np.nan
    )

    collab_avg = (
        collaboration["average_streams"].iloc[0]
        if len(collaboration) > 0 else np.nan
    )

    solo_median = (
        solo["median_streams"].iloc[0]
        if len(solo) > 0 else np.nan
    )

    collab_median = (
        collaboration["median_streams"].iloc[0]
        if len(collaboration) > 0 else np.nan
    )

    # --------------------------------------------------------
    # TOP METRICS
    # --------------------------------------------------------

    col1, col2, col3, col4 = st.columns(4)

    with col1:
        st.metric(
            "🎤 Solo average streams",
            f"{solo_avg:,.0f}" if not np.isnan(solo_avg) else "N/A"
        )

    with col2:
        st.metric(
            "🤝 Collaboration average streams",
            f"{collab_avg:,.0f}" if not np.isnan(collab_avg) else "N/A"
        )

    with col3:
        st.metric(
            "🎤 Solo median",
            f"{solo_median:,.0f}" if not np.isnan(solo_median) else "N/A"
        )

    with col4:
        st.metric(
            "🤝 Collaboration median",
            f"{collab_median:,.0f}" if not np.isnan(collab_median) else "N/A"
        )

    # --------------------------------------------------------
    # MAIN 3D VISUAL
    # --------------------------------------------------------

    st.markdown(
        "### 🌌 The Collaboration Streaming Universe"
    )

    fig = go.Figure()

    # Individual tracks
    fig.add_trace(
        go.Scatter3d(

            x=df_q3["artist_count"],

            y=df_q3["released_year"],

            z=df_q3["log_streams"],

            mode="markers",

            name="Tracks",

            marker=dict(
                size=7,
                color=df_q3["log_streams"],
                colorscale="Turbo",
                opacity=0.75,
                colorbar=dict(
                    title="Log₁₀ Streams"
                )
            ),

            text=df_q3["track_name"],

            customdata=np.column_stack([
                df_q3["artist_name"],
                df_q3["artist_type"],
                df_q3["artist_count"],
                df_q3["released_year"],
                df_q3["streams"]
            ]),

            hovertemplate=
                "<b>🎵 %{text}</b><br><br>"
                "🎤 Artist(s): %{customdata[0]}<br>"
                "👥 Type: <b>%{customdata[1]}</b><br>"
                "👤 Artist Count: <b>%{customdata[2]:.0f}</b><br>"
                "📅 Release Year: %{customdata[3]:.0f}<br>"
                "▶️ Streams: <b>%{customdata[4]:,.0f}</b>"
                "<extra>TRACK</extra>"
        )
    )

    # --------------------------------------------------------
    # GROUP AVERAGE POINTS
    # --------------------------------------------------------

    %%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Spotify Analytics — Hackathon",
    page_icon="🎵",
    layout="wide"
)


# ============================================================
# HEADER
# ============================================================

st.title("🎵 Spotify Analytics — Interactive Hit Lab")

st.markdown(
    """
    ### Explore what makes Spotify hits successful
    Three interactive 3D investigations into **streaming success, tempo, and collaborations**.
    """
)


# ============================================================
# LOAD DATA
# ============================================================

FILE_PATH = "/content/spotify dataset final.xlsm"

df = pd.read_excel(FILE_PATH)


# ============================================================
# RENAME COLUMNS
# ============================================================

df.columns = [
    "track_name",
    "artist_name",
    "artist_count",
    "released_year",
    "released_month",
    "released_day",
    "in_spotify_playlists",
    "in_spotify_charts",
    "streams",
    "in_apple_playlists",
    "in_apple_charts",
    "in_deezer_playlists",
    "in_deezer_charts",
    "in_shazam_charts",
    "bpm",
    "key",
    "mode",
    "danceability",
    "valence",
    "energy",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "cover_url"
]


# ============================================================
# NUMERIC CLEANING
# ============================================================

numeric_columns = [
    "artist_count",
    "released_year",
    "released_month",
    "released_day",
    "in_spotify_playlists",
    "in_spotify_charts",
    "streams",
    "in_apple_playlists",
    "in_apple_charts",
    "in_deezer_playlists",
    "in_deezer_charts",
    "in_shazam_charts",
    "bpm",
    "danceability",
    "valence",
    "energy",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# ============================================================
# SIDEBAR NAVIGATION
# ============================================================

st.sidebar.title("🎛️ Spotify Hit Lab")

question = st.sidebar.radio(
    "Choose an investigation:",
    [
        "Question 1 — Hit DNA",
        "Question 2 — Tempo Evolution",
        "Question 3 — Solo vs Collaboration"
    ]
)


# ============================================================
# COMMON YEAR FILTER
# ============================================================

available_years = sorted(
    df["released_year"].dropna().astype(int).unique()
)

min_year = min(available_years)
max_year = max(available_years)

default_start = max(min_year, max_year - 9)

year_range = st.sidebar.slider(
    "📅 Release year range",
    min_value=min_year,
    max_value=max_year,
    value=(default_start, max_year),
    step=1
)


# ============================================================
# QUESTION 1
# ============================================================

if question == "Question 1 — Hit DNA":

    st.header("🧬 Question 1 — Spotify Hit DNA")

    st.markdown(
        """
        **What audio features most strongly correlate with high stream counts?**
        """
    )

    audio_features = [
        "danceability",
        "energy",
        "valence",
        "acousticness",
        "instrumentalness",
        "liveness",
        "speechiness",
        "bpm"
    ]

    df_q1 = df.dropna(
        subset=["streams"] + audio_features
    ).copy()

    df_q1 = df_q1[
        (df_q1["streams"] > 0) &
        (df_q1["released_year"] >= year_range[0]) &
        (df_q1["released_year"] <= year_range[1])
    ]

    df_q1["log_streams"] = np.log10(df_q1["streams"])

    correlations = (
        df_q1[audio_features + ["streams"]]
        .corr()["streams"]
        .drop("streams")
        .sort_values(key=lambda x: abs(x), ascending=False)
    )

    top_feature_1 = correlations.index[0]
    top_feature_2 = correlations.index[1]

    col1, col2, col3 = st.columns(3)

    with col1:
        st.metric(
            "🥇 Strongest relationship",
            top_feature_1.title(),
            f"{correlations[top_feature_1]:.3f}"
        )

    with col2:
        st.metric(
            "🥈 Second strongest",
            top_feature_2.title(),
            f"{correlations[top_feature_2]:.3f}"
        )

    with col3:
        st.metric(
            "🎵 Tracks analyzed",
            f"{len(df_q1):,}"
        )

    st.markdown("### 🌌 Rotate the 3D Spotify Hit Universe")

    feature_x = st.sidebar.selectbox(
        "X-axis audio feature",
        audio_features,
        index=audio_features.index(top_feature_1)
    )

    feature_y = st.sidebar.selectbox(
        "Y-axis audio feature",
        audio_features,
        index=audio_features.index(top_feature_2)
    )

    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=df_q1[feature_x],
            y=df_q1[feature_y],
            z=df_q1["log_streams"],

            mode="markers",

            marker=dict(
                size=7,
                color=df_q1["log_streams"],
                colorscale="Turbo",
                opacity=0.78,
                colorbar=dict(
                    title="Log₁₀ Streams"
                )
            ),

            text=df_q1["track_name"],

            customdata=np.column_stack([
                df_q1["artist_name"],
                df_q1["streams"],
                df_q1["released_year"],
                df_q1[feature_x],
                df_q1[feature_y]
            ]),

            hovertemplate=
                "<b>🎵 %{text}</b><br><br>"
                "🎤 Artist: %{customdata[0]}<br>"
                "📅 Year: %{customdata[2]:.0f}<br>"
                f"🎚️ {feature_x.title()}: <b>%{{customdata[3]:.1f}}</b><br>"
                f"🎚️ {feature_y.title()}: <b>%{{customdata[4]:.1f}}</b><br>"
                "▶️ Streams: <b>%{customdata[1]:,.0f}</b>"
                "<extra></extra>"
        )
    )

    fig.update_layout(
        title=f"Spotify Hit DNA — {feature_x.title()} × {feature_y.title()} × Streams",

        template="plotly_dark",

        height=800,

        scene=dict(
            xaxis_title=feature_x.title(),
            yaxis_title=feature_y.title(),
            zaxis_title="Log₁₀(Streams)",

            camera=dict(
                eye=dict(
                    x=1.6,
                    y=1.6,
                    z=1.3
                )
            )
        ),

        margin=dict(
            l=0,
            r=0,
            b=0,
            t=60
        )
    )

    st.plotly_chart(
        fig,
        width="stretch",
        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "displaylogo": False,
            "modeBarButtonsToAdd": [
                "resetCameraDefault3d",
                "resetCameraLastSave3d"
            ]
        }
    )

    with st.expander("🧠 What does this mean?"):

        st.write(
            f"""
            **{top_feature_1.title()}** has the strongest Pearson correlation
            with streams in this dataset at **{correlations[top_feature_1]:.3f}**.

            However, the relationship is weak, meaning audio features alone
            do not strongly explain streaming success.

            **Important:** correlation does not imply causation.
            """
        )


# ============================================================
# QUESTION 2
# ============================================================

elif question == "Question 2 — Tempo Evolution":

    st.header("🥁 Question 2 — Tempo Evolution")

    st.markdown(
        """
        **How has the average tempo of top hits changed over recent years?**
        """
    )

    top_percent = st.sidebar.select_slider(
        "🔥 Define a Top Hit",
        options=[5, 10, 15, 20, 25],
        value=10,
        format_func=lambda x: f"Top {x}% streamed tracks"
    )

    df_q2 = df[
        (df["released_year"] >= year_range[0]) &
        (df["released_year"] <= year_range[1])
    ].copy()

    df_q2 = df_q2.dropna(
        subset=["released_year", "streams", "bpm"]
    )

    df_q2 = df_q2[df_q2["streams"] > 0]
    df_q2 = df_q2[df_q2["bpm"] > 0]

    df_q2["stream_percentile"] = (
        df_q2
        .groupby("released_year")["streams"]
        .rank(
            pct=True,
            ascending=True
        )
    )

    df_q2["is_top_hit"] = (
        df_q2["stream_percentile"]
        >= (1 - top_percent / 100)
    )

    top_hits = df_q2[
        df_q2["is_top_hit"]
    ].copy()

    yearly = (
        top_hits
        .groupby("released_year")
        .agg(
            average_bpm=("bpm", "mean"),
            median_bpm=("bpm", "median"),
            tracks=("track_name", "count"),
            average_streams=("streams", "mean")
        )
        .reset_index()
        .sort_values("released_year")
    )

    top_hits["log_streams"] = np.log10(
        top_hits["streams"]
    )

    yearly["log_average_streams"] = np.log10(
        yearly["average_streams"]
    )

    yearly["rolling_bpm"] = (
        yearly["average_bpm"]
        .rolling(
            window=3,
            min_periods=1
        )
        .mean()
    )

    first_year = yearly.iloc[0]
    latest_year = yearly.iloc[-1]

    bpm_change = (
        latest_year["average_bpm"]
        - first_year["average_bpm"]
    )

    col1, col2, col3, col4 = st.columns(4)

    with col1:
        st.metric(
            "📅 First year",
            f"{int(first_year['released_year'])}"
        )

    with col2:
        st.metric(
            "🥁 First average BPM",
            f"{first_year['average_bpm']:.1f}"
        )

    with col3:
        st.metric(
            "🚀 Latest average BPM",
            f"{latest_year['average_bpm']:.1f}"
        )

    with col4:
        st.metric(
            "📈 BPM change",
            f"{bpm_change:+.1f}"
        )

    st.markdown(
        "### 🌌 Spin through the evolution of Spotify's top hits"
    )

    fig = go.Figure()

    # Individual tracks
    fig.add_trace(
        go.Scatter3d(
            x=top_hits["released_year"],
            y=top_hits["bpm"],
            z=top_hits["log_streams"],

            mode="markers",

            name="Top Hit Tracks",

            marker=dict(
                size=7,
                color=top_hits["bpm"],
                colorscale="Turbo",
                opacity=0.78,
                colorbar=dict(
                    title="BPM"
                )
            ),

            text=top_hits["track_name"],

            customdata=np.column_stack([
                top_hits["artist_name"],
                top_hits["streams"],
                top_hits["released_year"],
                top_hits["bpm"]
            ]),

            hovertemplate=
                "<b>🎵 %{text}</b><br><br>"
                "🎤 Artist: %{customdata[0]}<br>"
                "📅 Year: %{customdata[2]:.0f}<br>"
                "🥁 Tempo: <b>%{customdata[3]:.1f} BPM</b><br>"
                "▶️ Streams: <b>%{customdata[1]:,.0f}</b>"
                "<extra>TOP HIT</extra>"
        )
    )

    # Yearly average
    fig.add_trace(
        go.Scatter3d(
            x=yearly["released_year"],
            y=yearly["average_bpm"],
            z=yearly["log_average_streams"],

            mode="lines+markers",

            name="Average BPM",

            line=dict(
                width=10
            ),

            marker=dict(
                size=13,
                color=yearly["average_bpm"],
                colorscale="Turbo",
                showscale=False
            ),

            customdata=np.column_stack([
                yearly["tracks"],
                yearly["median_bpm"],
                yearly["rolling_bpm"],
                yearly["average_streams"]
            ]),

            hovertemplate=
                "<b>📅 %{x:.0f}</b><br><br>"
                "🔥 Average Tempo: <b>%{y:.1f} BPM</b><br>"
                "Median Tempo: %{customdata[1]:.1f} BPM<br>"
                "Top Hits: %{customdata[0]:.0f}<br>"
                "3-Year Rolling BPM: %{customdata[2]:.1f}<br>"
                "Average Streams: %{customdata[3]:,.0f}"
                "<extra>YEARLY AVERAGE</extra>"
        )
    )

    fig.update_layout(
        title="Spotify Top-Hit Tempo Evolution",

        template="plotly_dark",

        height=800,

        scene=dict(
            xaxis_title="Release Year",
            yaxis_title="BPM",
            zaxis_title="Log₁₀(Streams)",

            camera=dict(
                eye=dict(
                    x=1.6,
                    y=1.6,
                    z=1.3
                )
            )
        ),

        margin=dict(
            l=0,
            r=0,
            b=0,
            t=60
        )
    )

    st.plotly_chart(
        fig,
        width="stretch",
        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "displaylogo": False,
            "modeBarButtonsToAdd": [
                "resetCameraDefault3d",
                "resetCameraLastSave3d"
            ]
        }
    )

    if bpm_change > 2:
        insight = "📈 Top-hit tempo has generally increased."
    elif bpm_change < -2:
        insight = "📉 Top-hit tempo has generally decreased."
    else:
        insight = "➡️ Top-hit tempo has remained relatively stable."

    st.success(insight)

    with st.expander("🧠 Methodology"):
        st.write(
            f"""
            A **Top Hit** is defined as a track belonging to the highest
            **{top_percent}% of streamed tracks within its release year**.

            This prevents older years from automatically dominating the
            comparison simply because their tracks have had more time to
            accumulate streams.

            The connected line represents the yearly average BPM.
            """
        )


# ============================================================
# QUESTION 3
# ============================================================

else:

    st.header("🤝 Question 3 — Solo vs Collaboration")

    st.markdown(
        """
        **Do solo artists or collaborations tend to have higher streaming success?**
        """
    )

    # --------------------------------------------------------
    # FILTER DATA
    # --------------------------------------------------------

    df_q3 = df[
        (df["released_year"] >= year_range[0]) &
        (df["released_year"] <= year_range[1])
    ].copy()

    df_q3 = df_q3.dropna(
        subset=[
            "streams",
            "artist_count",
            "released_year"
        ]
    )

    df_q3 = df_q3[
        df_q3["streams"] > 0
    ]

    df_q3 = df_q3[
        df_q3["artist_count"] >= 1
    ]

    # --------------------------------------------------------
    # CLASSIFY SOLO / COLLABORATION
    # --------------------------------------------------------

    df_q3["artist_type"] = np.where(
        df_q3["artist_count"] == 1,
        "Solo",
        "Collaboration"
    )

    df_q3["log_streams"] = np.log10(
        df_q3["streams"]
    )

    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    summary = (
        df_q3
        .groupby("artist_type")
        .agg(
            average_streams=("streams", "mean"),
            median_streams=("streams", "median"),
            average_artist_count=("artist_count", "mean"),
            tracks=("track_name", "count"),
            average_log_streams=("log_streams", "mean")
        )
        .reset_index()
    )

    solo = summary[
        summary["artist_type"] == "Solo"
    ]

    collaboration = summary[
        summary["artist_type"] == "Collaboration"
    ]

    solo_avg = (
        solo["average_streams"].iloc[0]
        if len(solo) > 0 else np.nan
    )

    collab_avg = (
        collaboration["average_streams"].iloc[0]
        if len(collaboration) > 0 else np.nan
    )

    solo_median = (
        solo["median_streams"].iloc[0]
        if len(solo) > 0 else np.nan
    )

    collab_median = (
        collaboration["median_streams"].iloc[0]
        if len(collaboration) > 0 else np.nan
    )

    # --------------------------------------------------------
    # TOP METRICS
    # --------------------------------------------------------

    col1, col2, col3, col4 = st.columns(4)

    with col1:
        st.metric(
            "🎤 Solo average streams",
            f"{solo_avg:,.0f}" if not np.isnan(solo_avg) else "N/A"
        )

    with col2:
        st.metric(
            "🤝 Collaboration average streams",
            f"{collab_avg:,.0f}" if not np.isnan(collab_avg) else "N/A"
        )

    with col3:
        st.metric(
            "🎤 Solo median",
            f"{solo_median:,.0f}" if not np.isnan(solo_median) else "N/A"
        )

    with col4:
        st.metric(
            "🤝 Collaboration median",
            f"{collab_median:,.0f}" if not np.isnan(collab_median) else "N/A"
        )

    # --------------------------------------------------------
    # MAIN 3D VISUAL
    # --------------------------------------------------------

    st.markdown(
        "### 🌌 The Collaboration Streaming Universe"
    )

    fig = go.Figure()

    # Individual tracks
    fig.add_trace(
        go.Scatter3d(

            x=df_q3["artist_count"],

            y=df_q3["released_year"],

            z=df_q3["log_streams"],

            mode="markers",

            name="Tracks",

            marker=dict(
                size=7,
                color=df_q3["log_streams"],
                colorscale="Turbo",
                opacity=0.75,
                colorbar=dict(
                    title="Log₁₀ Streams"
                )
            ),

            text=df_q3["track_name"],

            customdata=np.column_stack([
                df_q3["artist_name"],
                df_q3["artist_type"],
                df_q3["artist_count"],
                df_q3["released_year"],
                df_q3["streams"]
            ]),

            hovertemplate=
                "<b>🎵 %{text}</b><br><br>"
                "🎤 Artist(s): %{customdata[0]}<br>"
                "👥 Type: <b>%{customdata[1]}</b><br>"
                "👤 Artist Count: <b>%{customdata[2]:.0f}</b><br>"
                "📅 Release Year: %{customdata[3]:.0f}<br>"
                "▶️ Streams: <b>%{customdata[4]:,.0f}</b>"
                "<extra>TRACK</extra>"
        )
    )

    # --------------------------------------------------------
    # GROUP AVERAGE POINTS
    # --------------------------------------------------------

    group_average = (
        df_q3
        .groupby("artist_type")
        .agg(
            avg_artist_count=("artist_count", "mean"),
            avg_year=("released_year", "mean"),
            avg_streams=("streams", "mean"),
            median_streams=("streams", "median"),
            track_count=("track_name", "count")
        )
        .reset_index()
    )

    group_average["log_avg_streams"] = np.log10(
        group_average["avg_streams"]
    )

    fig.add_trace(
        go.Scatter3d(

            x=group_average["avg_artist_count"],

            y=group_average["avg_year"],

            z=group_average["log_avg_streams"],

            mode="markers+text",

            name="Group Average",

            marker=dict(
                size=22,
                color=[1, 2],
                colorscale="Turbo",
                opacity=1,
                line=dict(
                    width=3,
                    color="white"
                )
            ),

            text=group_average["artist_type"],

            textposition="top center",

            customdata=np.column_stack([
                group_average["avg_streams"],
                group_average["median_streams"],
                group_average["track_count"]
            ]),

            hovertemplate=
                "<b>%{text}</b><br><br>"
                "⭐ Average Streams: <b>%{customdata[0]:,.0f}</b><br>"
                "📊 Median Streams: %{customdata[1]:,.0f}<br>"
                "🎵 Tracks: %{customdata[2]:,.0f}"
                "<extra>GROUP AVERAGE</extra>"
        )
    )

    # --------------------------------------------------------
    # LAYOUT
    # --------------------------------------------------------

    fig.update_layout(

        title="Solo Artists vs Collaborations — Streaming Success",

        template="plotly_dark",

        height=850,

        scene=dict(

            xaxis_title="Number of Artists",

            yaxis_title="Release Year",

            zaxis_title="Log₁₀(Streams)",

            camera=dict(
                eye=dict(
                    x=1.7,
                    y=1.7,
                    z=1.4
                )
            )
        ),

        margin=dict(
            l=0,
            r=0,
            b=0,
            t=60
        ),

        legend=dict(
            x=0.02,
            y=0.98
        )
    )

    st.plotly_chart(
        fig,

        width="stretch",

        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "displaylogo": False,

            "modeBarButtonsToAdd": [
                "resetCameraDefault3d",
                "resetCameraLastSave3d"
            ]
        }
    )

    # ========================================================
    # AUTOMATIC CONCLUSION
    # ========================================================

    if not np.isnan(solo_avg) and not np.isnan(collab_avg):

        if solo_avg > collab_avg:

            difference = (
                (solo_avg - collab_avg)
                / collab_avg
                * 100
            )

            st.success(
                f"🎤 **Solo artists have the higher average streaming count "
                f"in this dataset**, by approximately **{difference:.1f}%**."
            )

        else:

            difference = (
                (collab_avg - solo_avg)
                / solo_avg
                * 100
            )

            st.success(
                f"🤝 **Collaborations have the higher average streaming count "
                f"in this dataset**, by approximately **{difference:.1f}%**."
            )

    # --------------------------------------------------------
    # METHODOLOGY
    # --------------------------------------------------------

    with st.expander("🧠 Methodology — How Q3 is calculated"):

        st.write(
            """
            **Solo artist:** `artist_count = 1`

            **Collaboration:** `artist_count > 1`

            Streaming success is measured using the number of Spotify streams.

            Because stream counts are highly skewed, the 3D height uses
            **log₁₀(streams)**. This makes the distribution easier to see
            without changing the underlying streaming values.

            The large markers represent the average streaming performance
            of each artist type.

            **Important:** A higher average does not prove that collaborating
            causes higher streaming success. Other factors such as artist
            popularity, release year, genre, promotion, and playlist exposure
            may also influence streams.
            """
        )

    # --------------------------------------------------------
    # SUMMARY TABLE
    # --------------------------------------------------------

    st.markdown("### 📊 Streaming Success Summary")

    display_summary = summary[
        [
            "artist_type",
            "tracks",
            "average_streams",
            "median_streams",
            "average_artist_count"
        ]
    ].copy()

    display_summary.columns = [
        "Artist Type",
        "Tracks",
        "Average Streams",
        "Median Streams",
        "Average Artist Count"
    ]

    st.dataframe(
        display_summary,
        width="stretch",
        hide_index=True
    )


# ============================================================
# FOOTER
# ============================================================

st.markdown("---")

st.caption(
    "🎵 Spotify Hit Lab • Interactive Plotly 3D Analysis • Hackathon Edition"
)

    # --------------------------------------------------------
    # LAYOUT
    # --------------------------------------------------------

    fig.update_layout(

        title="Solo Artists vs Collaborations — Streaming Success",

        template="plotly_dark",

        height=850,

        scene=dict(

            xaxis_title="Number of Artists",

            yaxis_title="Release Year",

            zaxis_title="Log₁₀(Streams)",

            camera=dict(
                eye=dict(
                    x=1.7,
                    y=1.7,
                    z=1.4
                )
            )
        ),

        margin=dict(
            l=0,
            r=0,
            b=0,
            t=60
        ),

        legend=dict(
            x=0.02,
            y=0.98
        )
    )

    st.plotly_chart(
        fig,

        width="stretch",

        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "displaylogo": False,

            "modeBarButtonsToAdd": [
                "resetCameraDefault3d",
                "resetCameraLastSave3d"
            ]
        }
    )

    # ========================================================
    # AUTOMATIC CONCLUSION
    # ========================================================

    if not np.isnan(solo_avg) and not np.isnan(collab_avg):

        if solo_avg > collab_avg:

            difference = (
                (solo_avg - collab_avg)
                / collab_avg
                * 100
            )

            st.success(
                f"🎤 **Solo artists have the higher average streaming count "
                f"in this dataset**, by approximately **{difference:.1f}%**."
            )

        else:

            difference = (
                (collab_avg - solo_avg)
                / solo_avg
                * 100
            )

            st.success(
                f"🤝 **Collaborations have the higher average streaming count "
                f"in this dataset**, by approximately **{difference:.1f}%**."
            )

    # --------------------------------------------------------
    # METHODOLOGY
    # --------------------------------------------------------

    with st.expander("🧠 Methodology — How Q3 is calculated"):

        st.write(
            """
            **Solo artist:** `artist_count = 1`

            **Collaboration:** `artist_count > 1`

            Streaming success is measured using the number of Spotify streams.

            Because stream counts are highly skewed, the 3D height uses
            **log₁₀(streams)**. This makes the distribution easier to see
            without changing the underlying streaming values.

            The large markers represent the average streaming performance
            of each artist type.

            **Important:** A higher average does not prove that collaborating
            causes higher streaming success. Other factors such as artist
            popularity, release year, genre, promotion, and playlist exposure
            may also influence streams.
            """
        )

    # --------------------------------------------------------
    # SUMMARY TABLE
    # --------------------------------------------------------

    st.markdown("### 📊 Streaming Success Summary")

    display_summary = summary[
        [
            "artist_type",
            "tracks",
            "average_streams",
            "median_streams",
            "average_artist_count"
        ]
    ].copy()

    display_summary.columns = [
        "Artist Type",
        "Tracks",
        "Average Streams",
        "Median Streams",
        "Average Artist Count"
    ]

    st.dataframe(
        display_summary,
        width="stretch",
        hide_index=True
    )


# ============================================================
# FOOTER
# ============================================================

st.markdown("---")

st.caption(
    "🎵 Spotify Hit Lab • Interactive Plotly 3D Analysis • Hackathon Edition"
)

Overwriting app.py


In [47]:
!pkill -f streamlit || true

^C


In [48]:
!streamlit run app.py \
    --server.address 0.0.0.0 \
    --server.port 8501 \
    --server.headless true \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    > /content/streamlit.log 2>&1 &

In [49]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Mon, 14 Sep 2026 07:58:21 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7459
last-modified: Mon, 14 Sep 2026 06:51:37 GMT
etag: "cbbfe301ca782e74ffed1e90134ffd01"
cache-control: no-cache



In [52]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Spotify Hit Lab",
    page_icon="🎵",
    layout="wide"
)


# ============================================================
# TITLE
# ============================================================

st.title("🎵 Spotify Hit Lab")

st.markdown(
    """
    ### Interactive 3D Spotify Analysis
    Explore the patterns behind streaming success, tempo evolution,
    and solo artists versus collaborations.
    """
)


# ============================================================
# LOAD DATA
# ============================================================

FILE_PATH = "/content/spotify dataset final.xlsm"

try:
    df = pd.read_excel(FILE_PATH)
except Exception as e:
    st.error(f"Could not load the dataset: {e}")
    st.stop()


# ============================================================
# RENAME COLUMNS
# ============================================================

expected_columns = [
    "track_name",
    "artist_name",
    "artist_count",
    "released_year",
    "released_month",
    "released_day",
    "in_spotify_playlists",
    "in_spotify_charts",
    "streams",
    "in_apple_playlists",
    "in_apple_charts",
    "in_deezer_playlists",
    "in_deezer_charts",
    "in_shazam_charts",
    "bpm",
    "key",
    "mode",
    "danceability",
    "valence",
    "energy",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "cover_url"
]

if len(df.columns) == len(expected_columns):
    df.columns = expected_columns
else:
    st.error(
        f"Dataset has {len(df.columns)} columns, "
        f"but the app expects {len(expected_columns)} columns."
    )
    st.stop()


# ============================================================
# NUMERIC CLEANING
# ============================================================

numeric_columns = [
    "artist_count",
    "released_year",
    "released_month",
    "released_day",
    "in_spotify_playlists",
    "in_spotify_charts",
    "streams",
    "in_apple_playlists",
    "in_apple_charts",
    "in_deezer_playlists",
    "in_deezer_charts",
    "in_shazam_charts",
    "bpm",
    "danceability",
    "valence",
    "energy",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.title("🎛️ Spotify Hit Lab")

question = st.sidebar.radio(
    "Choose a question:",
    [
        "Question 1 — Hit DNA",
        "Question 2 — Tempo Evolution",
        "Question 3 — Solo vs Collaboration"
    ]
)


# ============================================================
# COMMON YEAR INFORMATION
# ============================================================

available_years = sorted(
    df["released_year"]
    .dropna()
    .astype(int)
    .unique()
)

if len(available_years) == 0:
    st.error("No valid release years were found.")
    st.stop()

min_year = min(available_years)
max_year = max(available_years)

default_start = max(
    min_year,
    max_year - 9
)

year_range = st.sidebar.slider(
    "📅 Release year range",
    min_value=min_year,
    max_value=max_year,
    value=(default_start, max_year),
    step=1
)


# ============================================================
# QUESTION 1
# ============================================================

if question == "Question 1 — Hit DNA":

    st.header("🧬 Question 1 — Spotify Hit DNA")

    st.markdown(
        """
        **What audio features most strongly correlate with high stream counts?**
        """
    )

    audio_features = [
        "danceability",
        "energy",
        "valence",
        "acousticness",
        "instrumentalness",
        "liveness",
        "speechiness",
        "bpm"
    ]

    # --------------------------------------------------------
    # FILTER
    # --------------------------------------------------------

    df_q1 = df[
        (df["released_year"] >= year_range[0]) &
        (df["released_year"] <= year_range[1])
    ].copy()

    df_q1 = df_q1.dropna(
        subset=["streams"] + audio_features
    )

    df_q1 = df_q1[
        df_q1["streams"] > 0
    ]

    # --------------------------------------------------------
    # LOG STREAMS
    # --------------------------------------------------------

    df_q1["log_streams"] = np.log10(
        df_q1["streams"]
    )

    # --------------------------------------------------------
    # CORRELATIONS
    # --------------------------------------------------------

    correlations = (
        df_q1[audio_features + ["streams"]]
        .corr()["streams"]
        .drop("streams")
        .sort_values(
            key=lambda x: abs(x),
            ascending=False
        )
    )

    top_feature_1 = correlations.index[0]
    top_feature_2 = correlations.index[1]

    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    col1, col2, col3 = st.columns(3)

    with col1:
        st.metric(
            "🥇 Strongest relationship",
            top_feature_1.title(),
            f"{correlations[top_feature_1]:.3f}"
        )

    with col2:
        st.metric(
            "🥈 Second strongest",
            top_feature_2.title(),
            f"{correlations[top_feature_2]:.3f}"
        )

    with col3:
        st.metric(
            "🎵 Tracks analyzed",
            f"{len(df_q1):,}"
        )

    # --------------------------------------------------------
    # AXIS CONTROLS
    # --------------------------------------------------------

    st.markdown(
        "### 🌌 Rotate the Spotify Hit Universe"
    )

    feature_x = st.sidebar.selectbox(
        "Q1 — X-axis feature",
        audio_features,
        index=audio_features.index(top_feature_1)
    )

    feature_y = st.sidebar.selectbox(
        "Q1 — Y-axis feature",
        audio_features,
        index=audio_features.index(top_feature_2)
    )

    # --------------------------------------------------------
    # 3D FIGURE
    # --------------------------------------------------------

    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(

            x=df_q1[feature_x],

            y=df_q1[feature_y],

            z=df_q1["log_streams"],

            mode="markers",

            name="Spotify Tracks",

            marker=dict(
                size=7,
                color=df_q1["log_streams"],
                colorscale="Turbo",
                opacity=0.78,
                colorbar=dict(
                    title="Log₁₀ Streams"
                )
            ),

            text=df_q1["track_name"],

            customdata=np.column_stack([
                df_q1["artist_name"],
                df_q1["streams"],
                df_q1["released_year"],
                df_q1[feature_x],
                df_q1[feature_y]
            ]),

            hovertemplate=
                "<b>🎵 %{text}</b><br><br>"
                "🎤 Artist: %{customdata[0]}<br>"
                "📅 Year: %{customdata[2]:.0f}<br>"
                f"🎚️ {feature_x.title()}: "
                "<b>%{customdata[3]:.1f}</b><br>"
                f"🎚️ {feature_y.title()}: "
                "<b>%{customdata[4]:.1f}</b><br>"
                "▶️ Streams: "
                "<b>%{customdata[1]:,.0f}</b>"
                "<extra></extra>"
        )
    )

    fig.update_layout(

        title=(
            f"Spotify Hit DNA — "
            f"{feature_x.title()} × "
            f"{feature_y.title()} × Streams"
        ),

        template="plotly_dark",

        height=800,

        scene=dict(

            xaxis_title=feature_x.title(),

            yaxis_title=feature_y.title(),

            zaxis_title="Log₁₀(Streams)",

            camera=dict(
                eye=dict(
                    x=1.6,
                    y=1.6,
                    z=1.3
                )
            )
        ),

        margin=dict(
            l=0,
            r=0,
            b=0,
            t=60
        )
    )

    st.plotly_chart(
        fig,
        width="stretch",
        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "displaylogo": False,
            "modeBarButtonsToAdd": [
                "resetCameraDefault3d",
                "resetCameraLastSave3d"
            ]
        }
    )

    # --------------------------------------------------------
    # CONCLUSION
    # --------------------------------------------------------

    with st.expander("🧠 Q1 — What does this mean?"):

        st.write(
            f"""
            **{top_feature_1.title()}** has the strongest Pearson
            correlation with streams in the selected dataset:

            **r = {correlations[top_feature_1]:.3f}**

            The relationship is relatively weak, suggesting that
            audio features alone do not strongly explain streaming
            success.

            **Important:** correlation does not imply causation.
            """
        )


# ============================================================
# QUESTION 2
# ============================================================

elif question == "Question 2 — Tempo Evolution":

    st.header("🥁 Question 2 — Tempo Evolution")

    st.markdown(
        """
        **How has the average tempo of top hits changed over recent years?**
        """
    )

    # --------------------------------------------------------
    # TOP HIT CONTROL
    # --------------------------------------------------------

    top_percent = st.sidebar.select_slider(
        "🔥 Define a Top Hit",

        options=[
            5,
            10,
            15,
            20,
            25
        ],

        value=10,

        format_func=lambda x:
            f"Top {x}% streamed tracks"
    )

    # --------------------------------------------------------
    # FILTER
    # --------------------------------------------------------

    df_q2 = df[
        (df["released_year"] >= year_range[0]) &
        (df["released_year"] <= year_range[1])
    ].copy()

    df_q2 = df_q2.dropna(
        subset=[
            "released_year",
            "streams",
            "bpm"
        ]
    )

    df_q2 = df_q2[
        (df_q2["streams"] > 0) &
        (df_q2["bpm"] > 0)
    ]

    # --------------------------------------------------------
    # IDENTIFY TOP HITS WITHIN EACH YEAR
    # --------------------------------------------------------

    df_q2["stream_percentile"] = (
        df_q2
        .groupby("released_year")["streams"]
        .rank(
            pct=True,
            ascending=True
        )
    )

    df_q2["is_top_hit"] = (
        df_q2["stream_percentile"]
        >= (1 - top_percent / 100)
    )

    top_hits = df_q2[
        df_q2["is_top_hit"]
    ].copy()

    # --------------------------------------------------------
    # YEARLY SUMMARY
    # --------------------------------------------------------

    yearly = (
        top_hits
        .groupby("released_year")
        .agg(
            average_bpm=("bpm", "mean"),
            median_bpm=("bpm", "median"),
            tracks=("track_name", "count"),
            average_streams=("streams", "mean")
        )
        .reset_index()
        .sort_values("released_year")
    )

    if len(yearly) == 0:
        st.warning(
            "Not enough data for the selected filters."
        )
        st.stop()

    top_hits["log_streams"] = np.log10(
        top_hits["streams"]
    )

    yearly["log_average_streams"] = np.log10(
        yearly["average_streams"]
    )

    yearly["rolling_bpm"] = (
        yearly["average_bpm"]
        .rolling(
            window=3,
            min_periods=1
        )
        .mean()
    )

    # --------------------------------------------------------
    # CHANGE
    # --------------------------------------------------------

    first_year = yearly.iloc[0]
    latest_year = yearly.iloc[-1]

    bpm_change = (
        latest_year["average_bpm"]
        - first_year["average_bpm"]
    )

    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    col1, col2, col3, col4 = st.columns(4)

    with col1:
        st.metric(
            "📅 First year",
            f"{int(first_year['released_year'])}"
        )

    with col2:
        st.metric(
            "🥁 First average BPM",
            f"{first_year['average_bpm']:.1f}"
        )

    with col3:
        st.metric(
            "🚀 Latest average BPM",
            f"{latest_year['average_bpm']:.1f}"
        )

    with col4:
        st.metric(
            "📈 BPM change",
            f"{bpm_change:+.1f}"
        )

    # --------------------------------------------------------
    # 3D GRAPH
    # --------------------------------------------------------

    st.markdown(
        "### 🌌 Spin through the evolution of top hits"
    )

    fig = go.Figure()

    # Individual tracks
    fig.add_trace(
        go.Scatter3d(

            x=top_hits["released_year"],

            y=top_hits["bpm"],

            z=top_hits["log_streams"],

            mode="markers",

            name="Top Hit Tracks",

            marker=dict(
                size=7,
                color=top_hits["bpm"],
                colorscale="Turbo",
                opacity=0.78,
                colorbar=dict(
                    title="BPM"
                )
            ),

            text=top_hits["track_name"],

            customdata=np.column_stack([
                top_hits["artist_name"],
                top_hits["streams"],
                top_hits["released_year"],
                top_hits["bpm"]
            ]),

            hovertemplate=
                "<b>🎵 %{text}</b><br><br>"
                "🎤 Artist: %{customdata[0]}<br>"
                "📅 Year: %{customdata[2]:.0f}<br>"
                "🥁 Tempo: "
                "<b>%{customdata[3]:.1f} BPM</b><br>"
                "▶️ Streams: "
                "<b>%{customdata[1]:,.0f}</b>"
                "<extra>TOP HIT</extra>"
        )
    )

    # Yearly average
    fig.add_trace(
        go.Scatter3d(

            x=yearly["released_year"],

            y=yearly["average_bpm"],

            z=yearly["log_average_streams"],

            mode="lines+markers",

            name="Average BPM",

            line=dict(
                width=10
            ),

            marker=dict(
                size=13,
                color=yearly["average_bpm"],
                colorscale="Turbo",
                showscale=False
            ),

            customdata=np.column_stack([
                yearly["tracks"],
                yearly["median_bpm"],
                yearly["rolling_bpm"],
                yearly["average_streams"]
            ]),

            hovertemplate=
                "<b>📅 %{x:.0f}</b><br><br>"
                "🔥 Average Tempo: "
                "<b>%{y:.1f} BPM</b><br>"
                "Median Tempo: "
                "%{customdata[1]:.1f} BPM<br>"
                "Top Hits: "
                "%{customdata[0]:.0f}<br>"
                "3-Year Rolling BPM: "
                "%{customdata[2]:.1f}<br>"
                "Average Streams: "
                "%{customdata[3]:,.0f}"
                "<extra>YEARLY AVERAGE</extra>"
        )
    )

    fig.update_layout(

        title="Spotify Top-Hit Tempo Evolution",

        template="plotly_dark",

        height=800,

        scene=dict(

            xaxis_title="Release Year",

            yaxis_title="BPM",

            zaxis_title="Log₁₀(Streams)",

            camera=dict(
                eye=dict(
                    x=1.6,
                    y=1.6,
                    z=1.3
                )
            )
        ),

        margin=dict(
            l=0,
            r=0,
            b=0,
            t=60
        )
    )

    st.plotly_chart(
        fig,
        width="stretch",
        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "displaylogo": False,
            "modeBarButtonsToAdd": [
                "resetCameraDefault3d",
                "resetCameraLastSave3d"
            ]
        }
    )

    # --------------------------------------------------------
    # AUTOMATIC INSIGHT
    # --------------------------------------------------------

    if bpm_change > 2:

        st.success(
            "📈 Top-hit tempo has generally increased "
            "over the selected period."
        )

    elif bpm_change < -2:

        st.success(
            "📉 Top-hit tempo has generally decreased "
            "over the selected period."
        )

    else:

        st.info(
            "➡️ Top-hit tempo has remained relatively stable "
            "over the selected period."
        )

    # --------------------------------------------------------
    # METHODOLOGY
    # --------------------------------------------------------

    with st.expander("🧠 Q2 — Methodology"):

        st.write(
            f"""
            A **Top Hit** is defined as a track belonging to the
            highest **{top_percent}% of streamed tracks within its
            release year**.

            This prevents older years from automatically dominating
            the comparison because older tracks have had more time
            to accumulate streams.

            The connected 3D line represents the yearly average BPM.
            """
        )


# ============================================================
# QUESTION 3
# ============================================================

else:

    st.header("🤝 Question 3 — Solo vs Collaboration")

    st.markdown(
        """
        **Do solo artists or collaborations tend to have higher
        streaming success?**
        """
    )

    # --------------------------------------------------------
    # FILTER
    # --------------------------------------------------------

    df_q3 = df[
        (df["released_year"] >= year_range[0]) &
        (df["released_year"] <= year_range[1])
    ].copy()

    df_q3 = df_q3.dropna(
        subset=[
            "streams",
            "artist_count",
            "released_year"
        ]
    )

    df_q3 = df_q3[
        (df_q3["streams"] > 0) &
        (df_q3["artist_count"] >= 1)
    ]

    if len(df_q3) == 0:
        st.warning(
            "Not enough data for the selected filters."
        )
        st.stop()

    # --------------------------------------------------------
    # CLASSIFY
    # --------------------------------------------------------

    df_q3["artist_type"] = np.where(
        df_q3["artist_count"] == 1,
        "Solo",
        "Collaboration"
    )

    df_q3["log_streams"] = np.log10(
        df_q3["streams"]
    )

    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    summary = (
        df_q3
        .groupby("artist_type")
        .agg(
            average_streams=("streams", "mean"),
            median_streams=("streams", "median"),
            average_artist_count=("artist_count", "mean"),
            tracks=("track_name", "count")
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # GET VALUES SAFELY
    # --------------------------------------------------------

    solo_rows = summary[
        summary["artist_type"] == "Solo"
    ]

    collab_rows = summary[
        summary["artist_type"] == "Collaboration"
    ]

    if len(solo_rows) > 0:
        solo_avg = solo_rows["average_streams"].iloc[0]
        solo_median = solo_rows["median_streams"].iloc[0]
    else:
        solo_avg = np.nan
        solo_median = np.nan

    if len(collab_rows) > 0:
        collab_avg = collab_rows["average_streams"].iloc[0]
        collab_median = collab_rows["median_streams"].iloc[0]
    else:
        collab_avg = np.nan
        collab_median = np.nan

    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    col1, col2, col3, col4 = st.columns(4)

    with col1:
        if not np.isnan(solo_avg):
            st.metric(
                "🎤 Solo average streams",
                f"{solo_avg:,.0f}"
            )
        else:
            st.metric(
                "🎤 Solo average streams",
                "N/A"
            )

    with col2:
        if not np.isnan(collab_avg):
            st.metric(
                "🤝 Collaboration average streams",
                f"{collab_avg:,.0f}"
            )
        else:
            st.metric(
                "🤝 Collaboration average streams",
                "N/A"
            )

    with col3:
        if not np.isnan(solo_median):
            st.metric(
                "🎤 Solo median streams",
                f"{solo_median:,.0f}"
            )
        else:
            st.metric(
                "🎤 Solo median streams",
                "N/A"
            )

    with col4:
        if not np.isnan(collab_median):
            st.metric(
                "🤝 Collaboration median streams",
                f"{collab_median:,.0f}"
            )
        else:
            st.metric(
                "🤝 Collaboration median streams",
                "N/A"
            )

    # --------------------------------------------------------
    # 3D VISUAL
    # --------------------------------------------------------

    st.markdown(
        "### 🌌 The Collaboration Streaming Universe"
    )

    fig = go.Figure()

    # --------------------------------------------------------
    # INDIVIDUAL TRACKS
    # --------------------------------------------------------

    fig.add_trace(
        go.Scatter3d(

            x=df_q3["artist_count"],

            y=df_q3["released_year"],

            z=df_q3["log_streams"],

            mode="markers",

            name="Tracks",

            marker=dict(
                size=7,
                color=df_q3["log_streams"],
                colorscale="Turbo",
                opacity=0.75,
                colorbar=dict(
                    title="Log₁₀ Streams"
                )
            ),

            text=df_q3["track_name"],

            customdata=np.column_stack([
                df_q3["artist_name"],
                df_q3["artist_type"],
                df_q3["artist_count"],
                df_q3["released_year"],
                df_q3["streams"]
            ]),

            hovertemplate=
                "<b>🎵 %{text}</b><br><br>"
                "🎤 Artist(s): %{customdata[0]}<br>"
                "👥 Type: "
                "<b>%{customdata[1]}</b><br>"
                "👤 Artist Count: "
                "<b>%{customdata[2]:.0f}</b><br>"
                "📅 Release Year: "
                "%{customdata[3]:.0f}<br>"
                "▶️ Streams: "
                "<b>%{customdata[4]:,.0f}</b>"
                "<extra>TRACK</extra>"
        )
    )

    # --------------------------------------------------------
    # GROUP AVERAGES
    # --------------------------------------------------------

    group_average = (
        df_q3
        .groupby("artist_type")
        .agg(
            avg_artist_count=("artist_count", "mean"),
            avg_year=("released_year", "mean"),
            avg_streams=("streams", "mean"),
            median_streams=("streams", "median"),
            track_count=("track_name", "count")
        )
        .reset_index()
    )

    group_average["log_avg_streams"] = np.log10(
        group_average["avg_streams"]
    )

    # --------------------------------------------------------
    # GROUP AVERAGE MARKERS
    # --------------------------------------------------------

    fig.add_trace(
        go.Scatter3d(

            x=group_average["avg_artist_count"],

            y=group_average["avg_year"],

            z=group_average["log_avg_streams"],

            mode="markers+text",

            name="Group Average",

            marker=dict(
                size=22,
                opacity=1,
                line=dict(
                    width=3,
                    color="white"
                )
            ),

            text=group_average["artist_type"],

            textposition="top center",

            customdata=np.column_stack([
                group_average["avg_streams"],
                group_average["median_streams"],
                group_average["track_count"]
            ]),

            hovertemplate=
                "<b>%{text}</b><br><br>"
                "⭐ Average Streams: "
                "<b>%{customdata[0]:,.0f}</b><br>"
                "📊 Median Streams: "
                "%{customdata[1]:,.0f}<br>"
                "🎵 Tracks: "
                "%{customdata[2]:,.0f}"
                "<extra>GROUP AVERAGE</extra>"
        )
    )

    # --------------------------------------------------------
    # 3D LAYOUT
    # --------------------------------------------------------

    fig.update_layout(

        title="Solo Artists vs Collaborations — Streaming Success",

        template="plotly_dark",

        height=850,

        scene=dict(

            xaxis_title="Number of Artists",

            yaxis_title="Release Year",

            zaxis_title="Log₁₀(Streams)",

            camera=dict(
                eye=dict(
                    x=1.7,
                    y=1.7,
                    z=1.4
                )
            )
        ),

        margin=dict(
            l=0,
            r=0,
            b=0,
            t=60
        ),

        legend=dict(
            x=0.02,
            y=0.98
        )
    )

    # --------------------------------------------------------
    # INTERACTIVE PLOT
    # --------------------------------------------------------

    st.plotly_chart(
        fig,

        width="stretch",

        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "displaylogo": False,
            "modeBarButtonsToAdd": [
                "resetCameraDefault3d",
                "resetCameraLastSave3d"
            ]
        }
    )

    # --------------------------------------------------------
    # AUTOMATIC CONCLUSION
    # --------------------------------------------------------

    if (
        not np.isnan(solo_avg)
        and not np.isnan(collab_avg)
    ):

        if solo_avg > collab_avg:

            difference = (
                (solo_avg - collab_avg)
                / collab_avg
                * 100
            )

            st.success(
                f"🎤 **Solo artists have the higher average "
                f"streaming count** in the selected period, "
                f"by approximately **{difference:.1f}%**."
            )

        elif collab_avg > solo_avg:

            difference = (
                (collab_avg - solo_avg)
                / solo_avg
                * 100
            )

            st.success(
                f"🤝 **Collaborations have the higher average "
                f"streaming count** in the selected period, "
                f"by approximately **{difference:.1f}%**."
            )

        else:

            st.info(
                "➡️ Solo artists and collaborations have "
                "the same average streaming count."
            )

    # --------------------------------------------------------
    # MEDIAN INSIGHT
    # --------------------------------------------------------

    if (
        not np.isnan(solo_median)
        and not np.isnan(collab_median)
    ):

        if collab_median > solo_median:

            st.info(
                "📊 The collaboration group also has a higher "
                "median stream count, suggesting the pattern "
                "is not driven only by a few extremely popular tracks."
            )

        elif solo_median > collab_median:

            st.info(
                "📊 Solo artists also have a higher median "
                "stream count, suggesting the pattern persists "
                "beyond the average."
            )

    # --------------------------------------------------------
    # METHODOLOGY
    # --------------------------------------------------------

    with st.expander(
        "🧠 Q3 — Methodology"
    ):

        st.write(
            """
            **Solo artist:** `artist_count = 1`

            **Collaboration:** `artist_count > 1`

            Streaming success is measured using Spotify stream counts.

            Because streams are highly skewed, the vertical dimension
            uses **log₁₀(streams)**. This makes very large differences
            easier to visualize.

            Individual tracks are shown as floating 3D points.

            The large markers represent the average streaming performance
            of the two artist categories.

            The median is also reported because averages can be strongly
            affected by extremely popular tracks.

            **Important:** A higher streaming count does not prove that
            being solo or collaborating causes greater success. Other
            factors such as artist popularity, release year, genre,
            promotion, and playlist exposure can influence streams.
            """
        )

    # --------------------------------------------------------
    # SUMMARY TABLE
    # --------------------------------------------------------

    st.markdown(
        "### 📊 Streaming Success Summary"
    )

    display_summary = summary[
        [
            "artist_type",
            "tracks",
            "average_streams",
            "median_streams",
            "average_artist_count"
        ]
    ].copy()

    display_summary.columns = [
        "Artist Type",
        "Tracks",
        "Average Streams",
        "Median Streams",
        "Average Artist Count"
    ]

    st.dataframe(
        display_summary,
        width="stretch",
        hide_index=True
    )


# ============================================================
# FOOTER
# ============================================================

st.markdown("---")

st.caption(
    "🎵 Spotify Hit Lab • Interactive 3D Plotly Analysis • Hackathon Edition"
)

Overwriting app.py


In [53]:
!head -10 app.py


import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go


# ============================================================
# PAGE CONFIG
# ============================================================


In [54]:
!pkill -f streamlit || true

^C


In [55]:
!streamlit run app.py \
    --server.address 0.0.0.0 \
    --server.port 8501 \
    --server.headless true \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    > /content/streamlit.log 2>&1 &

In [56]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Mon, 14 Sep 2026 08:05:32 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7459
last-modified: Mon, 14 Sep 2026 06:51:37 GMT
etag: "cbbfe301ca782e74ffed1e90134ffd01"
cache-control: no-cache



In [57]:
from google.colab import output

output.serve_kernel_port_as_iframe(
    8501,
    height=950
)

<IPython.core.display.Javascript object>

In [58]:
!ls


 app.py       'spotify dataset final (1).xlsm'	 streamlit.log
 sample_data  'spotify dataset final.xlsm'


In [59]:
%%writefile requirements.txt
streamlit
pandas
numpy
plotly

Writing requirements.txt


In [60]:
!ls


 app.py		   'spotify dataset final (1).xlsm'
 requirements.txt  'spotify dataset final.xlsm'
 sample_data	    streamlit.log
